# Solari Hub Demand Forecasting — Daily `OrderVolume` per Micro-Fulfillment Hub

**Competition task.** Forecast daily `OrderVolume` for each of 1,115 Solari hubs over a *future*
date window. Scored with **RMSLE**, so the model is built and validated entirely in
`log1p` space.

**Data (provided files only — no external data, per `Rules.md`):**

| File | Role |
| --- | --- |
| `orders_train.csv` | 2013-01-01 → 2015-06-19 daily history incl. `OrderVolume`, `AppSessions` |
| `orders_test.csv` | 2015-06-20 → 2015-07-31 (42 days x 1,115 hubs = 46,830 rows), target hidden |
| `hub_metadata.csv` | static/semi-static hub attributes, joined on `HubID` |
| `sample_submission.csv` | required output format: `Id,OrderVolume` |

**Approach summary**

1. Gradient-boosted trees (XGBoost, `reg:squarederror` on `log1p(OrderVolume)`) — no deep
   learning, as permitted by the rules.
2. **Leakage-safe features.** Every historical aggregate (hub means, hub x weekday means,
   trailing-window means, ...) is recomputed *from the statistics window only*, which always
   ends strictly before the first target date of the block it feeds. The same code path
   produces validation features and test features, so the validation score is an honest
   estimate of the leaderboard score.
3. **Validation = a held-out future block** of exactly 42 days (2015-05-09 → 2015-06-19),
   mirroring the test horizon, plus an earlier 42-day back-test block for stability.
4. **Both Kaggle T4 GPUs are used**: the seed/ablation ensemble is dispatched as independent
   XGBoost jobs across `cuda:0` and `cuda:1` via worker subprocesses, with live `tqdm`
   progress bars driven by the workers' streamed boosting metrics.
5. Closed hubs (`IsOpen == 0`) are forced to a prediction of 0 — in the training history
   `IsOpen == 0` implies `OrderVolume == 0` without exception (verified in the notebook).

Everything the run produces is collected into a single downloadable ZIP in the final cell.

## 1. Environment and configuration

This cell defines every knob the notebook uses, in one place, so a re-run is fully described
by this dictionary. It performs no heavy work.

* **Inputs:** none (constants only).
* **Outputs:** the `CFG` namespace, the output directory tree under `WORK_DIR`, and a
  `RunLogger` that mirrors messages to stdout and to `logs/run.log` for the final archive.
* **Design decisions:**
  * `SEED = 42` is the single disclosed master seed required by `Rules.md`; per-model seeds
    are deterministic offsets from it.
  * `VALID_DAYS = 42` exactly matches the test horizon length, so the validation block has
    the same forecast-difficulty profile as the leaderboard block.
  * `WORK_DIR` defaults to `/kaggle/working` when present and otherwise to a local folder,
    which keeps the notebook runnable against Kaggle's read-only `input` mount.
  * Training rows are restricted to `IsOpen == 1 and OrderVolume > 0`: closed days are
    deterministic zeros and the handful of open-but-zero rows are data artefacts that would
    otherwise dominate an RMSLE objective.

In [ ]:
import os
import sys
import json
import time
import shutil
import platform
import datetime as dt
from pathlib import Path
from types import SimpleNamespace

# ----------------------------------------------------------------------------------
# Master configuration. Everything downstream reads from CFG; nothing is hard-coded.
# ----------------------------------------------------------------------------------
CFG = SimpleNamespace(
    SEED=42,                     # disclosed master seed (Rules.md #6)
    TARGET="OrderVolume",
    ID_COL="Id",
    KEY="HubID",
    DATE="Date",

    # ---- validation strategy -------------------------------------------------
    VALID_DAYS=42,               # == test horizon (2015-06-20 .. 2015-07-31)
    MIN_STATS_DAYS=180,          # statistics window must cover at least this span

    # ---- model ---------------------------------------------------------------
    N_SEEDS=4,                   # final ensemble members (spread over both GPUs)
    NUM_BOOST_ROUND=8000,
    EARLY_STOPPING=300,
    FINAL_ROUND_INFLATION=1.10,  # full-data refit uses 10% more rounds than val-optimal
    XGB_PARAMS={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",          # == RMSLE because the target is log1p-transformed
        "tree_method": "hist",
        "max_depth": 10,
        "min_child_weight": 8.0,
        "eta": 0.03,
        "subsample": 0.90,
        "colsample_bytree": 0.70,
        "colsample_bylevel": 0.80,
        "reg_lambda": 1.5,
        "reg_alpha": 0.0,
        "max_bin": 256,
    },

    # ---- feature engineering -------------------------------------------------
    TRAIL_WINDOWS=(7, 28, 91, 182, 365),   # trailing-mean windows (days, stats-window only)
    SMOOTH_PRIOR=12.0,                     # shrinkage weight for group means
    RECENCY_HALFLIFE_DAYS=365.0,           # sample-weight half-life (ablated in Section 9)

    # ---- calibration ---------------------------------------------------------
    CALIB_GRID=[round(0.940 + 0.002 * i, 3) for i in range(86)],  # 0.940 .. 1.110

    # ---- runtime -------------------------------------------------------------
    WORKER_POLL_SEC=0.2,
    MAX_GPUS=2,
)

RUN_STARTED = dt.datetime.now()
RUN_TAG = RUN_STARTED.strftime("%Y%m%d_%H%M%S")

# Kaggle writes to /kaggle/working; fall back to the CWD elsewhere so the notebook is portable.
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "working"
OUT = SimpleNamespace(root=WORK_DIR / "solari_forecast")
OUT.models = OUT.root / "models"
OUT.preds = OUT.root / "predictions"
OUT.metrics = OUT.root / "metrics"
OUT.figures = OUT.root / "figures"
OUT.logs = OUT.root / "logs"
OUT.cache = OUT.root / "cache"          # intermediate parquet handed to the GPU workers
OUT.reports = OUT.root / "reports"
OUT.submission = OUT.root / "submission"
for _p in vars(OUT).values():
    Path(_p).mkdir(parents=True, exist_ok=True)


class RunLogger:
    """Minimal tee logger: stdout for the notebook, a file for the final ZIP."""

    def __init__(self, path):
        self.path = Path(path)
        self.path.write_text(f"# Solari forecasting run {RUN_TAG}\n", encoding="utf-8")

    def __call__(self, msg, level="INFO"):
        stamp = dt.datetime.now().strftime("%H:%M:%S")
        line = f"[{stamp}] {level:<5} {msg}"
        print(line, flush=True)
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(line + "\n")


LOG = RunLogger(OUT.logs / "run.log")
LOG(f"Run tag        : {RUN_TAG}")
LOG(f"Working dir    : {WORK_DIR}")
LOG(f"Artifact root  : {OUT.root}")
LOG(f"Master seed    : {CFG.SEED}")

## 2. Imports, reproducibility and hardware discovery

* **Purpose:** import the scientific stack, pin every random source to `CFG.SEED`, and detect
  how many CUDA devices XGBoost can actually use.
* **What it does:** sets `PYTHONHASHSEED`, seeds `random` and `numpy`, configures matplotlib
  for headless rendering, then probes each GPU through XGBoost itself (one throw-away
  boosting round per device) instead of trusting `nvidia-smi` alone. A device joins
  `GPU_DEVICES` only if real GPU training succeeded on it.
* **Outputs:** `GPU_DEVICES` (typically `["cuda:0", "cuda:1"]` on a Kaggle T4 x2 session) and
  `metrics/environment.json` recording library versions, hardware and seeds.
* **Design decision:** with no GPU the notebook degrades to CPU `hist` instead of failing —
  results stay identical in expectation, the run is just slower.

In [ ]:
import random
import warnings
import subprocess
import multiprocessing as mp

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")                 # headless-safe; figures are saved, then displayed
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

import xgboost as xgb
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "figure.autolayout": True,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 10,
})


def seed_everything(seed):
    """Pin every random source this notebook can influence."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


seed_everything(CFG.SEED)


def probe_gpus(max_gpus):
    """Return the CUDA devices on which XGBoost can genuinely train."""
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=120,
        )
        if smi.returncode != 0:
            LOG("nvidia-smi returned non-zero; assuming no CUDA devices.", "WARN")
            return []
        rows = [r.strip() for r in smi.stdout.strip().splitlines() if r.strip()]
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError) as exc:
        LOG(f"nvidia-smi unavailable ({exc}); running on CPU.", "WARN")
        return []

    devices = []
    for row in rows[:max_gpus]:
        dev = f"cuda:{row.split(',')[0].strip()}"
        try:
            x = np.random.rand(64, 4).astype(np.float32)
            y = np.random.rand(64).astype(np.float32)
            xgb.train({"device": dev, "tree_method": "hist", "verbosity": 0},
                      xgb.DMatrix(x, label=y), num_boost_round=1)
            devices.append(dev)
            LOG(f"GPU OK  -> {dev}  ({row})")
        except Exception as exc:                      # noqa: BLE001 - reason must be visible
            LOG(f"GPU probe failed for {dev}: {exc}", "WARN")
    return devices


GPU_DEVICES = probe_gpus(CFG.MAX_GPUS)
DEVICES = GPU_DEVICES if GPU_DEVICES else ["cpu"]
LOG(f"Training devices: {DEVICES}")

ENVIRONMENT = {
    "run_tag": RUN_TAG,
    "started_at": RUN_STARTED.isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": mp.cpu_count(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "xgboost": xgb.__version__,
    "matplotlib": matplotlib.__version__,
    "devices": DEVICES,
    "gpu_devices_detected": GPU_DEVICES,
    "seed": CFG.SEED,
}
(OUT.metrics / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
for _k, _v in ENVIRONMENT.items():
    LOG(f"  {_k:<22}: {_v}")

## 3. Dataset discovery

* **Purpose:** locate the four CSVs without hard-coding a Kaggle dataset slug, which differs
  between attached datasets and competition mounts.
* **What it does:** searches `/kaggle/input` recursively (then a few local fallbacks) for each
  required file name and returns the first match, preferring shallower paths so a
  competition mount wins over a nested copy. Raises immediately with a directory listing if
  anything is missing — no silent fallback to partial data.
* **Inputs:** the Kaggle filesystem. **Outputs:** `PATHS`, a dict of resolved `Path` objects.

In [ ]:
REQUIRED_FILES = ["orders_train.csv", "orders_test.csv", "hub_metadata.csv", "sample_submission.csv"]

SEARCH_ROOTS = [Path("/kaggle/input"), Path.cwd(), Path.cwd() / "data", Path.cwd().parent / "data"]


def discover_inputs(required=REQUIRED_FILES, roots=SEARCH_ROOTS):
    """Resolve each required CSV to a concrete path, preferring the shallowest match."""
    found, searched = {}, []
    for root in roots:
        if not root.exists():
            continue
        searched.append(str(root))
        for name in required:
            if name in found:
                continue
            matches = sorted(root.rglob(name), key=lambda p: (len(p.parts), str(p)))
            if matches:
                found[name] = matches[0]
    missing = [n for n in required if n not in found]
    if missing:
        listing = []
        for root in roots:
            if root.exists():
                listing += [f"  {p}" for p in sorted(root.rglob("*.csv"))[:40]]
        raise FileNotFoundError(
            "Could not locate: " + ", ".join(missing)
            + "\nSearched roots: " + ", ".join(searched)
            + "\nCSV files visible:\n" + ("\n".join(listing) or "  <none>")
            + "\nAttach the competition dataset to this notebook (Add Data) and re-run."
        )
    return found


PATHS = discover_inputs()
for _name, _path in PATHS.items():
    LOG(f"Found {_name:<22} -> {_path}  ({_path.stat().st_size / 1e6:.1f} MB)")

## 4. Loading and schema validation

* **Purpose:** read the CSVs with explicit dtypes and assert every structural assumption the
  rest of the notebook relies on, so a malformed input fails loudly here rather than silently
  corrupting a submission.
* **What it does:** parses `Date` to `datetime64`, downcasts small integer flags, then runs a
  battery of checks: required columns present, no duplicate `(HubID, Date)` keys, train/test
  date ranges disjoint and contiguous, every test hub present in train, every hub present in
  the metadata, submission `Id` set identical to the test `Id` set, and `IsOpen == 0 =>
  OrderVolume == 0`.
* **Inputs:** `PATHS`. **Outputs:** `train_raw`, `test_raw`, `hub_meta`, `sample_sub`, plus
  `metrics/data_validation.json` recording every check and its result.
* **Assumption made explicit:** `AppSessions` exists only in train. It is therefore **never**
  used as a row-level feature (it would be unavailable at prediction time); it is used solely
  through hub-level aggregates computed inside the statistics window.

In [ ]:
FLAG_DTYPES = {
    "HubID": "int32", "Weekday": "int8", "IsOpen": "int8",
    "PromoActive": "int8", "RegionalHoliday": "int8", "SchoolClosureFlag": "int8",
}


def read_orders(path, has_target):
    dtypes = dict(FLAG_DTYPES)
    if has_target:
        dtypes.update({"OrderVolume": "int32", "AppSessions": "int32"})
    else:
        dtypes.update({"Id": "int32"})
    df = pd.read_csv(path, dtype=dtypes, parse_dates=["Date"])
    return df.sort_values(["HubID", "Date"]).reset_index(drop=True)


train_raw = read_orders(PATHS["orders_train.csv"], has_target=True)
test_raw = read_orders(PATHS["orders_test.csv"], has_target=False)
hub_meta = pd.read_csv(PATHS["hub_metadata.csv"])
sample_sub = pd.read_csv(PATHS["sample_submission.csv"])

checks, failures = {}, []


def check(name, condition, detail=""):
    ok = bool(condition)
    checks[name] = {"passed": ok, "detail": str(detail)}
    if not ok:
        failures.append(f"{name}: {detail}")
    return ok


check("train_columns",
      set(["HubID", "Weekday", "Date", "OrderVolume", "AppSessions", "IsOpen",
           "PromoActive", "RegionalHoliday", "SchoolClosureFlag"]).issubset(train_raw.columns),
      sorted(train_raw.columns))
check("test_columns",
      set(["Id", "HubID", "Weekday", "Date", "IsOpen", "PromoActive",
           "RegionalHoliday", "SchoolClosureFlag"]).issubset(test_raw.columns),
      sorted(test_raw.columns))
check("train_key_unique", not train_raw.duplicated(["HubID", "Date"]).any(),
      int(train_raw.duplicated(["HubID", "Date"]).sum()))
check("test_key_unique", not test_raw.duplicated(["HubID", "Date"]).any(),
      int(test_raw.duplicated(["HubID", "Date"]).sum()))
check("test_after_train", test_raw["Date"].min() > train_raw["Date"].max(),
      f"train_max={train_raw['Date'].max().date()} test_min={test_raw['Date'].min().date()}")
check("test_contiguous",
      test_raw["Date"].nunique() == (test_raw["Date"].max() - test_raw["Date"].min()).days + 1,
      f"{test_raw['Date'].nunique()} distinct dates")
check("test_hubs_in_train", set(test_raw.HubID) <= set(train_raw.HubID),
      len(set(test_raw.HubID) - set(train_raw.HubID)))
check("hubs_in_metadata", set(train_raw.HubID) <= set(hub_meta.HubID),
      len(set(train_raw.HubID) - set(hub_meta.HubID)))
check("submission_ids_match", set(sample_sub[CFG.ID_COL]) == set(test_raw[CFG.ID_COL]),
      f"sample={len(sample_sub)} test={len(test_raw)}")
check("closed_implies_zero",
      int(((train_raw.IsOpen == 0) & (train_raw.OrderVolume > 0)).sum()) == 0,
      int(((train_raw.IsOpen == 0) & (train_raw.OrderVolume > 0)).sum()))
check("no_nulls_in_test", not test_raw.isna().any().any(),
      test_raw.isna().sum().to_dict())
check("target_non_negative", int((train_raw.OrderVolume < 0).sum()) == 0,
      int((train_raw.OrderVolume < 0).sum()))
check("appsessions_absent_from_test", "AppSessions" not in test_raw.columns,
      "AppSessions is train-only and is never used as a row-level feature")

TRAIN_MAX_DATE = train_raw["Date"].max()
TEST_MIN_DATE, TEST_MAX_DATE = test_raw["Date"].min(), test_raw["Date"].max()
TEST_HORIZON_DAYS = (TEST_MAX_DATE - TEST_MIN_DATE).days + 1

(OUT.metrics / "data_validation.json").write_text(
    json.dumps({"checks": checks,
                "train_rows": int(len(train_raw)), "test_rows": int(len(test_raw)),
                "hubs": int(train_raw.HubID.nunique()),
                "train_range": [str(train_raw.Date.min().date()), str(TRAIN_MAX_DATE.date())],
                "test_range": [str(TEST_MIN_DATE.date()), str(TEST_MAX_DATE.date())],
                "test_horizon_days": int(TEST_HORIZON_DAYS)}, indent=2),
    encoding="utf-8")

for _name, _res in checks.items():
    LOG(f"{'PASS' if _res['passed'] else 'FAIL'}  {_name:<28} {_res['detail']}")
if failures:
    raise AssertionError("Input validation failed:\n  - " + "\n  - ".join(failures))

LOG(f"train {len(train_raw):,} rows | test {len(test_raw):,} rows | hubs {train_raw.HubID.nunique()}")
LOG(f"train {train_raw.Date.min().date()} .. {TRAIN_MAX_DATE.date()} | "
    f"test {TEST_MIN_DATE.date()} .. {TEST_MAX_DATE.date()} ({TEST_HORIZON_DAYS} days)")
display(train_raw.head())
display(test_raw.head())
display(hub_meta.head())

## 5. Exploratory data analysis — target, calendar and operating status

* **Purpose:** establish the facts that justify the modelling choices, and persist them as
  figures/tables for the write-up required by `Rules.md` #7.
* **What it does:** summarises the target distribution on raw and `log1p` scales, quantifies
  closed days and open-but-zero rows, and plots (a) the network-wide daily order curve,
  (b) the raw vs `log1p` target histograms, (c) mean orders by weekday split by promotion,
  and (d) mean orders by month.
* **Inputs:** `train_raw`. **Outputs:** `figures/eda_*.png`, `metrics/target_summary.json`.
* **Why it matters:** the heavy right skew plus the order-of-magnitude spread across hubs is
  exactly the situation RMSLE is designed for, and confirms training in `log1p` space with an
  L2 objective is the metric-aligned choice.

In [ ]:
def save_fig(fig, name):
    """Persist a figure to figures/ and show it inline."""
    path = OUT.figures / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    LOG(f"figure -> {path.name}")
    plt.show()
    plt.close(fig)
    return path


open_rows = train_raw[train_raw.IsOpen == 1]
open_pos = open_rows[open_rows.OrderVolume > 0]

TARGET_SUMMARY = {
    "rows_total": int(len(train_raw)),
    "rows_closed": int((train_raw.IsOpen == 0).sum()),
    "pct_closed": float((train_raw.IsOpen == 0).mean() * 100),
    "rows_open_zero_volume": int(((train_raw.IsOpen == 1) & (train_raw.OrderVolume == 0)).sum()),
    "rows_used_for_training": int(len(open_pos)),
    "raw": {k: float(v) for k, v in open_pos.OrderVolume.describe().items()},
    "log1p": {k: float(v) for k, v in np.log1p(open_pos.OrderVolume).describe().items()},
    "raw_skew": float(open_pos.OrderVolume.skew()),
    "log1p_skew": float(np.log1p(open_pos.OrderVolume).skew()),
    "hub_mean_min": float(open_pos.groupby("HubID").OrderVolume.mean().min()),
    "hub_mean_max": float(open_pos.groupby("HubID").OrderVolume.mean().max()),
    "promo_share_train": float(train_raw.PromoActive.mean()),
    "promo_share_test": float(test_raw.PromoActive.mean()),
}
TARGET_SUMMARY["hub_mean_ratio_max_over_min"] = (
    TARGET_SUMMARY["hub_mean_max"] / TARGET_SUMMARY["hub_mean_min"])
(OUT.metrics / "target_summary.json").write_text(json.dumps(TARGET_SUMMARY, indent=2), encoding="utf-8")

LOG(f"closed rows                 : {TARGET_SUMMARY['rows_closed']:,} "
    f"({TARGET_SUMMARY['pct_closed']:.1f}%)")
LOG(f"open-but-zero rows (dropped): {TARGET_SUMMARY['rows_open_zero_volume']:,}")
LOG(f"training rows (open, > 0)   : {TARGET_SUMMARY['rows_used_for_training']:,}")
LOG(f"hub mean spread             : {TARGET_SUMMARY['hub_mean_min']:.0f} .. "
    f"{TARGET_SUMMARY['hub_mean_max']:.0f} "
    f"({TARGET_SUMMARY['hub_mean_ratio_max_over_min']:.1f}x) -> RMSLE is the right metric")
LOG(f"skew raw {TARGET_SUMMARY['raw_skew']:.2f} -> log1p {TARGET_SUMMARY['log1p_skew']:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

daily = train_raw.groupby("Date").OrderVolume.sum()
axes[0, 0].plot(daily.index, daily.values / 1e6, lw=0.7, color="#2b6cb0")
axes[0, 0].plot(daily.index, (daily / 1e6).rolling(28, center=True).mean(),
                lw=2, color="#c05621", label="28-day centred mean")
axes[0, 0].set_title("Network-wide daily order volume")
axes[0, 0].set_ylabel("millions of orders")
axes[0, 0].legend()

axes[0, 1].hist(open_pos.OrderVolume, bins=90, color="#2b6cb0", alpha=0.85)
axes[0, 1].set_title("Target distribution (open, non-zero) — raw")
axes[0, 1].set_xlabel("OrderVolume")
ax_t = axes[0, 1].twiny()
ax_t.hist(np.log1p(open_pos.OrderVolume), bins=90, color="#c05621", alpha=0.35)
ax_t.set_xlabel("log1p(OrderVolume)  [orange]")

dow = open_pos.groupby(["Weekday", "PromoActive"]).OrderVolume.mean().unstack()
dow.plot(kind="bar", ax=axes[1, 0], color=["#a0aec0", "#c05621"])
axes[1, 0].set_title("Mean orders by weekday x promotion (open hubs)")
axes[1, 0].set_xlabel("Weekday (1=Mon .. 7=Sun)")
axes[1, 0].legend(title="PromoActive")

mon = open_pos.assign(Month=open_pos.Date.dt.month).groupby("Month").OrderVolume.mean()
axes[1, 1].bar(mon.index, mon.values, color="#2b6cb0")
axes[1, 1].axvspan(5.5, 7.5, color="#c05621", alpha=0.15)
axes[1, 1].set_title("Mean orders by calendar month (test months shaded)")
axes[1, 1].set_xlabel("Month")

save_fig(fig, "eda_target_overview")
display(pd.DataFrame(TARGET_SUMMARY["raw"], index=["raw"]).T.join(
    pd.DataFrame(TARGET_SUMMARY["log1p"], index=["log1p"]).T))

### 5.1 Hub-level structure, renovation closures and the train/test gap

* **Purpose:** characterise the two structural quirks that most affect generalisation:
  (i) 181 hubs have a ~6-month block of *missing rows* (the renovation closures mentioned in
  the dataset description), and (ii) `RegionalHoliday` is identically `0` across the whole test
  window while it takes values 0–3 in train.
* **What it does:** counts rows per hub, detects calendar gaps per hub, compares flag
  distributions between the train tail and the test block, and plots the per-hub history
  coverage heat strip plus the promo/school-closure rate over time.
* **Outputs:** `figures/eda_coverage_and_shift.png`, `metrics/train_test_shift.json`,
  `RENOVATION_HUBS` (hub IDs with a gap of 30+ days).
* **Design consequence:** because holiday levels 1–3 never occur in the test window, the model
  must not lean on them; they are kept as features (they explain historical dips, which
  improves the hub-level signal) but the validation block is chosen to contain a realistic
  holiday mix so the effect is measured rather than assumed.

In [ ]:
rows_per_hub = train_raw.groupby("HubID").size()
full_span_days = (TRAIN_MAX_DATE - train_raw.Date.min()).days + 1

gap_records = []
for hub, grp in train_raw.groupby("HubID", sort=True)[["Date"]]:
    d = grp["Date"].sort_values()
    diffs = d.diff().dt.days
    big = diffs[diffs > 1]
    if len(big):
        gap_records.append({"HubID": hub, "n_gaps": int(len(big)),
                            "max_gap_days": int(big.max()),
                            "gap_start": str((d.loc[big.idxmax()] - pd.Timedelta(days=int(big.max()))).date()),
                            "gap_end": str(d.loc[big.idxmax()].date())})
gaps_df = pd.DataFrame(gap_records)
RENOVATION_HUBS = set(gaps_df.loc[gaps_df.max_gap_days >= 30, "HubID"]) if len(gaps_df) else set()

train_tail = train_raw[train_raw.Date > TRAIN_MAX_DATE - pd.Timedelta(days=CFG.VALID_DAYS)]
SHIFT = {
    "hubs_with_any_gap": int(len(gaps_df)),
    "hubs_with_renovation_gap_30d+": int(len(RENOVATION_HUBS)),
    "max_gap_days_observed": int(gaps_df.max_gap_days.max()) if len(gaps_df) else 0,
    "rows_per_hub_min": int(rows_per_hub.min()),
    "rows_per_hub_max": int(rows_per_hub.max()),
    "full_span_days": int(full_span_days),
    "open_rate": {"train_all": float(train_raw.IsOpen.mean()),
                  "train_tail_42d": float(train_tail.IsOpen.mean()),
                  "test": float(test_raw.IsOpen.mean())},
    "promo_rate": {"train_all": float(train_raw.PromoActive.mean()),
                   "train_tail_42d": float(train_tail.PromoActive.mean()),
                   "test": float(test_raw.PromoActive.mean())},
    "school_closure_rate": {"train_all": float(train_raw.SchoolClosureFlag.mean()),
                            "train_tail_42d": float(train_tail.SchoolClosureFlag.mean()),
                            "test": float(test_raw.SchoolClosureFlag.mean())},
    "regional_holiday_value_counts": {
        "train": {str(k): int(v) for k, v in train_raw.RegionalHoliday.value_counts().items()},
        "test": {str(k): int(v) for k, v in test_raw.RegionalHoliday.value_counts().items()}},
}
(OUT.metrics / "train_test_shift.json").write_text(json.dumps(SHIFT, indent=2), encoding="utf-8")

LOG(f"hubs with a >=30-day history gap (renovation): {len(RENOVATION_HUBS)}")
LOG(f"open rate   train={SHIFT['open_rate']['train_all']:.3f}  test={SHIFT['open_rate']['test']:.3f}")
LOG(f"promo rate  train={SHIFT['promo_rate']['train_all']:.3f}  test={SHIFT['promo_rate']['test']:.3f}")
LOG(f"school rate train={SHIFT['school_closure_rate']['train_all']:.3f}  "
    f"test={SHIFT['school_closure_rate']['test']:.3f}")
LOG(f"RegionalHoliday in test: {SHIFT['regional_holiday_value_counts']['test']} "
    f"(levels 1-3 never occur in the test window)")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].hist(rows_per_hub.values, bins=60, color="#2b6cb0")
axes[0, 0].set_title(f"Rows of history per hub (full span = {full_span_days} days)")
axes[0, 0].set_xlabel("rows")

hub_means = open_pos.groupby("HubID").OrderVolume.mean().sort_values()
axes[0, 1].plot(np.arange(len(hub_means)), hub_means.values, color="#2b6cb0")
axes[0, 1].set_yscale("log")
axes[0, 1].set_title("Hub mean daily orders, sorted (log scale)")
axes[0, 1].set_xlabel("hub rank")

cal = pd.concat([
    train_raw.assign(block="train")[["Date", "PromoActive", "SchoolClosureFlag", "IsOpen", "block"]],
    test_raw.assign(block="test")[["Date", "PromoActive", "SchoolClosureFlag", "IsOpen", "block"]],
])
rate = cal.groupby("Date")[["PromoActive", "SchoolClosureFlag", "IsOpen"]].mean()
axes[1, 0].plot(rate.index, rate.PromoActive, lw=0.8, label="PromoActive")
axes[1, 0].plot(rate.index, rate.SchoolClosureFlag, lw=0.8, label="SchoolClosureFlag")
axes[1, 0].plot(rate.index, rate.IsOpen, lw=0.8, label="IsOpen")
axes[1, 0].axvline(TEST_MIN_DATE, color="#c05621", ls="--", lw=1.5)
axes[1, 0].set_title("Network flag rates over time (dashed line = test start)")
axes[1, 0].legend(fontsize=8)

cov = (train_raw.assign(one=1)
       .pivot_table(index="HubID", columns="Date", values="one", fill_value=0)
       .reindex(columns=pd.date_range(train_raw.Date.min(), TRAIN_MAX_DATE), fill_value=0))
axes[1, 1].imshow(cov.values, aspect="auto", cmap="Blues", interpolation="nearest")
axes[1, 1].set_title("History coverage (dark = row present) — renovation gap visible")
axes[1, 1].set_xlabel("day index")
axes[1, 1].set_ylabel("hub index")
axes[1, 1].grid(False)

save_fig(fig, "eda_coverage_and_shift")
if len(gaps_df):
    display(gaps_df.sort_values("max_gap_days", ascending=False).head(8))

### 5.2 Hub metadata: cleaning and exploratory relationships

* **Purpose:** turn `hub_metadata.csv` into a strictly numeric, null-free frame and inspect how
  each attribute relates to demand.
* **What it does:**
  * `CompetitorDistance`: 3 nulls, filled with the network median and flagged
    (`CompetitorDistanceMissing`), plus a `log1p` transform because the raw metres span four
    orders of magnitude.
  * `CompetitorOpenSinceMonth/Year`: 354 nulls meaning "no known competitor opening"; kept as
    `NaN` for XGBoost's native missing-value handling and flagged with `CompetitorKnown`.
  * `LoyaltyProgramSinceWeek/Year/Interval`: null exactly when `LoyaltyProgram == 0`;
    `LoyaltyProgramInterval` is expanded into a 12-element month bitmap (handling the
    `"Sept"` spelling, which `%b` parsing would miss).
* **Outputs:** `hub_meta_clean`, `figures/eda_hub_metadata.png`,
  `metrics/hub_metadata_summary.json`.

In [ ]:
MONTH_ABBR = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
              "jul": 7, "aug": 8, "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12}


def parse_interval(value):
    """'Mar,Jun,Sept,Dec' -> {3, 6, 9, 12}; unknown/blank -> empty set."""
    if not isinstance(value, str) or not value.strip():
        return set()
    months = set()
    for tok in value.split(","):
        key = tok.strip().lower()
        if key in MONTH_ABBR:
            months.add(MONTH_ABBR[key])
        else:
            raise ValueError(f"Unrecognised month token in LoyaltyProgramInterval: {tok!r}")
    return months


hub_meta_clean = hub_meta.copy()
hub_meta_clean["CompetitorDistanceMissing"] = hub_meta_clean.CompetitorDistance.isna().astype("int8")
_median_dist = hub_meta_clean.CompetitorDistance.median()
hub_meta_clean["CompetitorDistance"] = hub_meta_clean.CompetitorDistance.fillna(_median_dist)
hub_meta_clean["LogCompetitorDistance"] = np.log1p(hub_meta_clean.CompetitorDistance)
hub_meta_clean["CompetitorKnown"] = hub_meta_clean.CompetitorOpenSinceYear.notna().astype("int8")

_intervals = hub_meta_clean.LoyaltyProgramInterval.map(parse_interval)
for m in range(1, 13):
    hub_meta_clean[f"LoyaltyMonth_{m}"] = _intervals.map(lambda s, m=m: int(m in s)).astype("int8")
hub_meta_clean["LoyaltyIntervalCount"] = _intervals.map(len).astype("int8")

for col in ["HubFormat", "AssortmentTier", "LoyaltyProgram"]:
    hub_meta_clean[col] = hub_meta_clean[col].astype("int8")
hub_meta_clean["HubID"] = hub_meta_clean.HubID.astype("int32")

assert hub_meta_clean.HubID.is_unique, "hub_metadata contains duplicate HubID rows"
assert hub_meta_clean[["CompetitorDistance", "LogCompetitorDistance"]].notna().all().all()

_join = open_pos.merge(hub_meta_clean, on="HubID", how="left")
META_SUMMARY = {
    "competitor_distance_nulls_filled": int(hub_meta_clean.CompetitorDistanceMissing.sum()),
    "competitor_distance_median_used": float(_median_dist),
    "hubs_without_competitor_date": int((hub_meta_clean.CompetitorKnown == 0).sum()),
    "hubs_in_loyalty_program": int((hub_meta_clean.LoyaltyProgram == 1).sum()),
    "mean_orders_by_hub_format": {str(k): float(v) for k, v in
                                  _join.groupby("HubFormat").OrderVolume.mean().items()},
    "mean_orders_by_assortment": {str(k): float(v) for k, v in
                                  _join.groupby("AssortmentTier").OrderVolume.mean().items()},
    "mean_orders_by_loyalty": {str(k): float(v) for k, v in
                               _join.groupby("LoyaltyProgram").OrderVolume.mean().items()},
    "hub_format_counts": {str(k): int(v) for k, v in hub_meta_clean.HubFormat.value_counts().items()},
}
(OUT.metrics / "hub_metadata_summary.json").write_text(json.dumps(META_SUMMARY, indent=2), encoding="utf-8")

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
_join.boxplot(column="OrderVolume", by="HubFormat", ax=axes[0, 0], grid=False)
axes[0, 0].set_title("Orders by HubFormat"); axes[0, 0].set_xlabel("HubFormat")
_join.boxplot(column="OrderVolume", by="AssortmentTier", ax=axes[0, 1], grid=False)
axes[0, 1].set_title("Orders by AssortmentTier"); axes[0, 1].set_xlabel("AssortmentTier")

dist_bins = pd.qcut(_join.CompetitorDistance, 10, duplicates="drop")
axes[1, 0].plot(np.arange(dist_bins.nunique()),
                _join.groupby(dist_bins, observed=True).OrderVolume.mean().values,
                marker="o", color="#2b6cb0")
axes[1, 0].set_title("Mean orders by competitor-distance decile")
axes[1, 0].set_xlabel("decile (0 = closest competitor)")

_join.boxplot(column="OrderVolume", by="LoyaltyProgram", ax=axes[1, 1], grid=False)
axes[1, 1].set_title("Orders by LoyaltyProgram participation")
axes[1, 1].set_xlabel("LoyaltyProgram")
fig.suptitle("")
save_fig(fig, "eda_hub_metadata")

LOG(f"competitor distance nulls filled with median {_median_dist:.0f} m: "
    f"{META_SUMMARY['competitor_distance_nulls_filled']}")
LOG(f"hubs with no competitor-opening date: {META_SUMMARY['hubs_without_competitor_date']}")
display(hub_meta_clean.head())

### 5.3 `AppSessions`: a train-only signal, and how it is safely exploited

* **Purpose:** quantify the relationship between `AppSessions` and `OrderVolume` and decide
  how to use a column that **does not exist in the test file**.
* **What it does:** measures the correlation in `log1p` space, computes per-hub
  orders-per-session intensity, and plots the joint distribution.
* **Design decision (leakage control):** using `AppSessions` row-wise would be impossible at
  inference time, and imputing it from the same-day target would leak. It is therefore used
  **only** as hub-level summary statistics (mean log sessions, mean orders-per-session,
  session volatility) computed inside the statistics window, which are legitimate static-ish
  descriptors of a hub's customer base.
* **Outputs:** `figures/eda_appsessions.png`, `metrics/appsessions_summary.json`.

In [ ]:
_ls = np.log1p(open_pos.AppSessions)
_lv = np.log1p(open_pos.OrderVolume)
_agg = open_pos.groupby("HubID").agg(order_sum=("OrderVolume", "sum"),
                                     session_sum=("AppSessions", "sum"),
                                     mean_sessions=("AppSessions", "mean"))
per_hub = pd.DataFrame({
    "orders_per_session": _agg.order_sum / _agg.session_sum.clip(lower=1),
    "mean_sessions": _agg.mean_sessions,
})

APP_SUMMARY = {
    "pearson_log_sessions_vs_log_orders": float(np.corrcoef(_ls, _lv)[0, 1]),
    "spearman_sessions_vs_orders": float(open_pos.AppSessions.corr(open_pos.OrderVolume, method="spearman")),
    "orders_per_session_mean": float(per_hub.orders_per_session.mean()),
    "orders_per_session_p05": float(per_hub.orders_per_session.quantile(0.05)),
    "orders_per_session_p95": float(per_hub.orders_per_session.quantile(0.95)),
    "usage": "hub-level aggregates only (column absent from orders_test.csv)",
}
(OUT.metrics / "appsessions_summary.json").write_text(json.dumps(APP_SUMMARY, indent=2), encoding="utf-8")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
_idx = np.random.default_rng(CFG.SEED).choice(len(open_pos), size=min(60000, len(open_pos)), replace=False)
axes[0].scatter(_ls.values[_idx], _lv.values[_idx], s=2, alpha=0.08, color="#2b6cb0")
axes[0].set_xlabel("log1p(AppSessions)"); axes[0].set_ylabel("log1p(OrderVolume)")
axes[0].set_title(f"r = {APP_SUMMARY['pearson_log_sessions_vs_log_orders']:.3f} (log space)")
axes[1].hist(per_hub.orders_per_session, bins=60, color="#2b6cb0")
axes[1].set_title("Per-hub orders per app session")
axes[2].scatter(per_hub.mean_sessions, hub_means.reindex(per_hub.index), s=6, alpha=0.5, color="#c05621")
axes[2].set_xlabel("hub mean AppSessions"); axes[2].set_ylabel("hub mean OrderVolume")
axes[2].set_title("Hub intensity relationship")
save_fig(fig, "eda_appsessions")

LOG(f"log-space correlation AppSessions ~ OrderVolume: "
    f"{APP_SUMMARY['pearson_log_sessions_vs_log_orders']:.3f} "
    f"(used only via hub-level, stats-window aggregates)")

## 6. Preprocessing — a single unified panel with calendar-sequence features

* **Purpose:** build one row-per-`(HubID, Date)` panel covering train **and** test, carrying
  every feature that can be derived *without* looking at the target.
* **What it does:**
  1. Concatenates train and test on the shared schema and marks each row with `is_test`.
  2. Reindexes every hub onto the complete daily calendar (2013-01-01 → 2015-07-31). This
     matters because 181 hubs are missing ~6 months of rows (renovation), and without
     reindexing a "days since" counter would silently jump across the gap. Synthetic filler
     days are marked `RowPresent = 0` and treated as closed.
  3. Derives sequence features from columns that are **known in advance for the test window**
     (`IsOpen`, `PromoActive`, `SchoolClosureFlag`, `RegionalHoliday` are all supplied in
     `orders_test.csv`): promo run length, days since/until a promo, days since/until a closed
     day, open-run length, school-closure run length, and neighbour-day flags.
  4. Adds pure calendar features and the metadata-derived competitor/loyalty timings.
* **Leakage note:** nothing here touches `OrderVolume` or `AppSessions`, so computing these
  across the combined panel is safe — it uses only the operating calendar that the business
  plans in advance, which is exactly what the test file provides.
* **Inputs:** `train_raw`, `test_raw`, `hub_meta_clean`. **Outputs:** `panel` (DataFrame,
  ~1.05 M rows) and `SEQ_FEATURES` (list of generated column names).

In [ ]:
CAL_COLS = ["HubID", "Date", "Weekday", "IsOpen", "PromoActive", "RegionalHoliday", "SchoolClosureFlag"]

CALENDAR_START = train_raw.Date.min()
CALENDAR_END = TEST_MAX_DATE
FULL_DATES = pd.date_range(CALENDAR_START, CALENDAR_END, freq="D")

with tqdm(total=6, desc="Building panel", unit="step") as bar:
    # 1) stack train + test on the common schema -------------------------------------------
    stacked = pd.concat(
        [train_raw[CAL_COLS].assign(is_test=np.int8(0), Id=np.int32(-1),
                                    OrderVolume=train_raw.OrderVolume.astype("float32"),
                                    AppSessions=train_raw.AppSessions.astype("float32")),
         test_raw[CAL_COLS].assign(is_test=np.int8(1), Id=test_raw.Id.astype("int32"),
                                   OrderVolume=np.float32("nan"),
                                   AppSessions=np.float32("nan"))],
        ignore_index=True)
    stacked["RowPresent"] = np.int8(1)
    bar.update(1)

    # 2) reindex every hub onto the complete daily calendar ---------------------------------
    hub_ids = np.sort(train_raw.HubID.unique())
    grid = pd.MultiIndex.from_product([hub_ids, FULL_DATES], names=["HubID", "Date"]).to_frame(index=False)
    panel = grid.merge(stacked, on=["HubID", "Date"], how="left")
    panel = panel.sort_values(["HubID", "Date"], kind="mergesort").reset_index(drop=True)
    filler = panel.RowPresent.isna()
    LOG(f"panel rows {len(panel):,} | synthetic filler days (missing history) {int(filler.sum()):,}")
    for col, fill in [("RowPresent", 0), ("IsOpen", 0), ("PromoActive", 0),
                      ("RegionalHoliday", 0), ("SchoolClosureFlag", 0), ("is_test", 0), ("Id", -1)]:
        panel[col] = panel[col].fillna(fill)
    panel["Weekday"] = panel.Date.dt.dayofweek + 1      # 1 = Monday .. 7 = Sunday (matches source)
    for col in ["RowPresent", "IsOpen", "PromoActive", "RegionalHoliday", "SchoolClosureFlag",
                "is_test", "Weekday"]:
        panel[col] = panel[col].astype("int8")
    panel["Id"] = panel.Id.astype("int32")
    panel["HubID"] = panel.HubID.astype("int32")
    bar.update(1)

    # 3) sequence helpers --------------------------------------------------------------------
    g_hub = panel.groupby("HubID", sort=False)
    row_idx = pd.Series(np.arange(len(panel), dtype="float64"), index=panel.index)

    def run_length(flag):
        """Length of the current consecutive run of `flag == 1`, 0 where the flag is off."""
        f = pd.Series(np.asarray(flag).astype("int8"), index=panel.index)
        change = f.ne(f.groupby(panel.HubID).shift()).cumsum()
        rl = f.groupby([panel.HubID, change]).cumcount() + 1
        return (rl.to_numpy() * f.to_numpy()).astype("int16")

    def days_since(flag):
        """Days since the most recent day with `flag == 1` (large sentinel before the first)."""
        marked = row_idx.where(flag.astype(bool))
        last = marked.groupby(panel.HubID).ffill()
        out = (row_idx - last).fillna(9999.0)
        return out.clip(upper=9999).astype("int16")

    def days_until(flag):
        """Days until the next day with `flag == 1` (large sentinel after the last)."""
        marked = row_idx.where(flag.astype(bool))
        nxt = marked.groupby(panel.HubID).bfill()
        out = (nxt - row_idx).fillna(9999.0)
        return out.clip(upper=9999).astype("int16")
    bar.update(1)

    # 4) promotion / closure / school-closure dynamics ---------------------------------------
    promo = panel.PromoActive
    closed = (1 - panel.IsOpen).astype("int8").rename("Closed")
    panel["PromoRunLength"] = run_length(promo)
    panel["DaysSincePromo"] = days_since(promo)
    panel["DaysUntilPromo"] = days_until(promo)
    panel["PromoYesterday"] = g_hub.PromoActive.shift(1).fillna(0).astype("int8")
    panel["PromoTomorrow"] = g_hub.PromoActive.shift(-1).fillna(0).astype("int8")
    panel["OpenRunLength"] = run_length(panel.IsOpen)
    panel["ClosedRunLength"] = run_length(closed)
    panel["DaysSinceClosed"] = days_since(closed)
    panel["DaysUntilClosed"] = days_until(closed)
    panel["ClosedYesterday"] = (1 - g_hub.IsOpen.shift(1).fillna(1)).astype("int8")
    panel["ClosedTomorrow"] = (1 - g_hub.IsOpen.shift(-1).fillna(1)).astype("int8")
    panel["SchoolClosureRunLength"] = run_length(panel.SchoolClosureFlag)
    panel["HolidayYesterday"] = g_hub.RegionalHoliday.shift(1).fillna(0).astype("int8")
    panel["HolidayTomorrow"] = g_hub.RegionalHoliday.shift(-1).fillna(0).astype("int8")
    bar.update(1)

    # 5) calendar features --------------------------------------------------------------------
    d = panel.Date.dt
    panel["Year"] = d.year.astype("int16")
    panel["Month"] = d.month.astype("int8")
    panel["Day"] = d.day.astype("int8")
    panel["WeekOfYear"] = d.isocalendar().week.to_numpy().astype("int8")
    panel["DayOfYear"] = d.dayofyear.astype("int16")
    panel["DaysSinceStart"] = (panel.Date - CALENDAR_START).dt.days.astype("int16")
    panel["IsMonthStart"] = (panel.Day <= 5).astype("int8")
    panel["IsMonthEnd"] = (panel.Day >= 26).astype("int8")
    panel["IsMonthMid"] = ((panel.Day > 12) & (panel.Day < 19)).astype("int8")
    panel["IsWeekend"] = (panel.Weekday >= 6).astype("int8")
    panel["SinDayOfYear"] = np.sin(2 * np.pi * panel.DayOfYear / 365.25).astype("float32")
    panel["CosDayOfYear"] = np.cos(2 * np.pi * panel.DayOfYear / 365.25).astype("float32")
    bar.update(1)

    # 6) hub metadata + competitor / loyalty timings -------------------------------------------
    meta_cols = (["HubID", "HubFormat", "AssortmentTier", "CompetitorDistance",
                  "LogCompetitorDistance", "CompetitorDistanceMissing", "CompetitorKnown",
                  "CompetitorOpenSinceMonth", "CompetitorOpenSinceYear", "LoyaltyProgram",
                  "LoyaltyProgramSinceWeek", "LoyaltyProgramSinceYear", "LoyaltyIntervalCount"]
                 + [f"LoyaltyMonth_{m}" for m in range(1, 13)])
    panel = panel.merge(hub_meta_clean[meta_cols], on="HubID", how="left", validate="many_to_one")

    panel["CompetitorOpenMonths"] = (
        12 * (panel.Year - panel.CompetitorOpenSinceYear) + (panel.Month - panel.CompetitorOpenSinceMonth)
    ).astype("float32")
    panel.loc[panel.CompetitorKnown == 0, "CompetitorOpenMonths"] = np.nan
    panel["CompetitorActive"] = (panel.CompetitorOpenMonths > 0).astype("int8")
    panel["CompetitorOpenMonths"] = panel.CompetitorOpenMonths.clip(-24, 200)

    panel["LoyaltyWeeks"] = (
        52.0 * (panel.Year - panel.LoyaltyProgramSinceYear) + (panel.WeekOfYear - panel.LoyaltyProgramSinceWeek)
    ).astype("float32")
    panel.loc[panel.LoyaltyProgram == 0, "LoyaltyWeeks"] = np.nan
    panel["LoyaltyActive"] = ((panel.LoyaltyProgram == 1) & (panel.LoyaltyWeeks > 0)).astype("int8")
    panel["LoyaltyWeeks"] = panel.LoyaltyWeeks.clip(-52, 400)
    month_bitmap = panel[[f"LoyaltyMonth_{m}" for m in range(1, 13)]].to_numpy()
    panel["IsLoyaltyRoundMonth"] = (
        panel.LoyaltyActive.to_numpy() * month_bitmap[np.arange(len(panel)), panel.Month.to_numpy() - 1]
    ).astype("int8")
    panel.drop(columns=[f"LoyaltyMonth_{m}" for m in range(1, 13)], inplace=True)
    bar.update(1)

SEQ_FEATURES = ["PromoRunLength", "DaysSincePromo", "DaysUntilPromo", "PromoYesterday", "PromoTomorrow",
                "OpenRunLength", "ClosedRunLength", "DaysSinceClosed", "DaysUntilClosed",
                "ClosedYesterday", "ClosedTomorrow", "SchoolClosureRunLength",
                "HolidayYesterday", "HolidayTomorrow"]

LOG(f"panel shape {panel.shape}; memory {panel.memory_usage(deep=True).sum() / 1e6:.0f} MB")
assert panel.duplicated(["HubID", "Date"]).sum() == 0
assert int(panel.is_test.sum()) == len(test_raw), "test rows lost while building the panel"
display(panel.head(3))

## 7. Validation strategy — held-out *future* blocks, never a random split

* **Purpose:** define splits that reproduce the leaderboard situation exactly: predict a
  contiguous 42-day future block using only strictly-earlier history.
* **Blocks defined:**

  | Block | Statistics window (history) | Target window | Purpose |
  | --- | --- | --- | --- |
  | `backtest` | 2013-01-01 → val_start − 43 d | 42 days before `valid` | stability check on a second, earlier future block |
  | `valid` | 2013-01-01 → 2015-05-08 | 2015-05-09 → 2015-06-19 | model selection, early stopping, calibration |
  | `final` | 2013-01-01 → 2015-06-19 (all train) | 2015-06-20 → 2015-07-31 | leaderboard submission |

* **Why not K-fold:** rows are not exchangeable. A random split would place a hub's future
  days in the training fold alongside its neighbouring past days, letting the model memorise
  local level shifts and inflating the estimated score. Each block here has the same
  "history ends, then forecast 42 unseen days" structure as the real task.
* **Horizon parity:** every block is exactly `CFG.VALID_DAYS` (= the 42-day test horizon), so
  the average forecast distance from the last observation matches the leaderboard's.
* **Outputs:** the `BLOCKS` dict, `metrics/validation_blocks.json`, and a timeline figure.

In [ ]:
VALID_END = TRAIN_MAX_DATE
VALID_START = VALID_END - pd.Timedelta(days=CFG.VALID_DAYS - 1)
BACKTEST_END = VALID_START - pd.Timedelta(days=1)
BACKTEST_START = BACKTEST_END - pd.Timedelta(days=CFG.VALID_DAYS - 1)

BLOCKS = {
    "backtest": {"stats_end": BACKTEST_START - pd.Timedelta(days=1),
                 "target_start": BACKTEST_START, "target_end": BACKTEST_END, "scored": True},
    "valid":    {"stats_end": VALID_START - pd.Timedelta(days=1),
                 "target_start": VALID_START, "target_end": VALID_END, "scored": True},
    "final":    {"stats_end": TRAIN_MAX_DATE,
                 "target_start": TEST_MIN_DATE, "target_end": TEST_MAX_DATE, "scored": False},
}

for name, blk in BLOCKS.items():
    span = (blk["stats_end"] - CALENDAR_START).days + 1
    assert span >= CFG.MIN_STATS_DAYS, f"{name}: statistics window of {span} days is too short"
    assert blk["stats_end"] < blk["target_start"], f"{name}: statistics window overlaps the target"
    blk["stats_days"] = int(span)
    blk["target_days"] = int((blk["target_end"] - blk["target_start"]).days + 1)
    LOG(f"block {name:<9} history {CALENDAR_START.date()}..{blk['stats_end'].date()} "
        f"({span}d) -> target {blk['target_start'].date()}..{blk['target_end'].date()} "
        f"({blk['target_days']}d)")

assert BLOCKS["valid"]["target_days"] == TEST_HORIZON_DAYS == BLOCKS["final"]["target_days"], \
    "validation horizon must equal the test horizon"
assert BLOCKS["backtest"]["target_end"] < BLOCKS["valid"]["target_start"], "blocks must not overlap"

(OUT.metrics / "validation_blocks.json").write_text(json.dumps(
    {k: {kk: (str(vv.date()) if isinstance(vv, pd.Timestamp) else vv) for kk, vv in v.items()}
     for k, v in BLOCKS.items()}, indent=2), encoding="utf-8")

fig, ax = plt.subplots(figsize=(13, 3.0))
colors = {"backtest": "#805ad5", "valid": "#c05621", "final": "#2f855a"}
for i, (name, blk) in enumerate(BLOCKS.items()):
    ax.barh(i, (blk["stats_end"] - CALENDAR_START).days + 1, left=CALENDAR_START,
            height=0.45, color="#cbd5e0", edgecolor="none")
    ax.barh(i, blk["target_days"], left=blk["target_start"],
            height=0.45, color=colors[name], edgecolor="none")
ax.set_yticks(range(len(BLOCKS)))
ax.set_yticklabels(list(BLOCKS))
ax.set_xlim(CALENDAR_START, CALENDAR_END)
ax.set_title("Time-blocked validation design: grey = statistics/training history, "
             "colour = 42-day forecast target")
save_fig(fig, "validation_blocks")

## 8. Leakage-safe historical features

This is the core of the model. The test block is a contiguous 42-day window with **no target
values at all**, so classic lag features (`y[t-1]`, `y[t-7]`, ...) are unavailable without
recursive forecasting, which compounds error badly over a six-week horizon. Instead every row
gets *summary statistics of its hub's history*, always computed from a window that ends
**strictly before** the first day of the window being predicted.

* **Purpose:** map a `(hub, date)` row to "what this hub's demand looked like — overall, by
  weekday, by promo state and over several trailing windows — as of the last day we were
  allowed to see".
* **What `build_history_features(panel, stats_end, target_start, target_end)` does:**
  1. **Hub level:** count, mean / median / std / min / max / quartiles / IQR of
     `log1p(OrderVolume)`, mean and std of `log1p(AppSessions)`, orders-per-session intensity,
     coefficient of variation, open / promo / school-closure rates, Saturday and Sunday open
     rates, and days since the hub last traded.
  2. **Shrunk conditional means:** hub x weekday, hub x promo, hub x weekday x promo,
     hub x month, hub x school-closure. Each is shrunk toward its hub mean with an
     equivalent-sample-size prior, `m = (sum + k*hub_mean) / (n + k)` with `k =
     CFG.SMOOTH_PRIOR`, so thin cells (a hub that rarely trades on Sundays) cannot produce
     wild estimates.
  3. **Deltas:** each conditional mean also appears as a difference from the hub mean. Trees
     split far more efficiently on "this weekday runs 0.12 log-units above the hub's norm"
     than on two highly correlated absolute levels.
  4. **Trailing windows:** mean `log1p` volume over the last 7 / 28 / 91 / 182 / 365 days,
     their deltas and observation counts, trailing weekday and promo shapes, and three
     recent-vs-long-run trend terms. These carry the hub's *current level*, which is what a
     42-day-ahead forecast needs most.
  5. **Horizon:** `HorizonDays` = days from `stats_end` to the row's date, so the model can
     learn how fast to discount recent-level features as the forecast reaches further out.
* **Guarantees asserted in code:** `stats.Date.max() < target.Date.min()` for every call, and
  no row-level `OrderVolume` / `AppSessions` value is ever placed in the feature matrix.
* **Inputs:** `panel` and a window specification. **Outputs:** a feature `DataFrame`, plus the
  ordered `FEATURES` list that every block must reproduce identically.

In [ ]:
def _smooth(group_sum, group_cnt, fallback, prior=CFG.SMOOTH_PRIOR):
    """Shrink a group mean toward `fallback` using an equivalent-sample-size prior."""
    return (np.asarray(group_sum, dtype="float64") + prior * np.asarray(fallback, dtype="float64")) \
        / (np.asarray(group_cnt, dtype="float64") + prior)


def _grouped_mean(stats, keys, value_col, hub_fallback, name, prior=CFG.SMOOTH_PRIOR):
    """Shrunk group mean of `value_col` over `keys`, returned as a flat frame."""
    agg = stats.groupby(keys, observed=True)[value_col].agg(["sum", "count"])
    fb = agg.index.get_level_values("HubID").map(hub_fallback).to_numpy(dtype="float64")
    agg[name] = _smooth(agg["sum"].to_numpy(), agg["count"].to_numpy(), fb, prior)
    return agg[[name]].reset_index()


def build_history_features(panel_df, stats_end, target_start, target_end, tag, progress=False):
    """Feature matrix for the rows in [target_start, target_end], using history <= stats_end."""
    stats = panel_df[(panel_df.Date <= stats_end) & (panel_df.RowPresent == 1)
                     & (panel_df.IsOpen == 1) & (panel_df.OrderVolume > 0)].copy()
    target = panel_df[(panel_df.Date >= target_start) & (panel_df.Date <= target_end)
                      & (panel_df.RowPresent == 1)].copy()

    # ---- hard leakage guards --------------------------------------------------------------
    assert len(stats) and len(target), f"{tag}: empty statistics or target frame"
    assert stats.Date.max() < target.Date.min(), (
        f"{tag}: statistics end {stats.Date.max()} is not strictly before "
        f"target start {target.Date.min()}")

    stats["LogVol"] = np.log1p(stats.OrderVolume.to_numpy())
    stats["LogSess"] = np.log1p(stats.AppSessions.to_numpy())
    bar = tqdm(total=4, desc=f"features[{tag}]", leave=False, disable=not progress)

    # ------------------------------------------------------------- 1) hub-level aggregates
    hub = stats.groupby("HubID").agg(
        HubLogMean=("LogVol", "mean"), HubLogStd=("LogVol", "std"),
        HubLogMedian=("LogVol", "median"), HubLogMin=("LogVol", "min"),
        HubLogMax=("LogVol", "max"), HubObs=("LogVol", "size"),
        HubLogSessMean=("LogSess", "mean"), HubLogSessStd=("LogSess", "std"),
        HubVolSum=("OrderVolume", "sum"), HubSessSum=("AppSessions", "sum"),
        HubLastTradeOrd=("DaysSinceStart", "max"),
    )
    q = stats.groupby("HubID")["LogVol"].quantile([0.25, 0.75]).unstack()
    hub["HubLogP25"], hub["HubLogP75"] = q[0.25], q[0.75]
    hub["HubLogIQR"] = q[0.75] - q[0.25]
    hub["HubOrdersPerSession"] = hub.HubVolSum / hub.HubSessSum.clip(lower=1.0)
    hub["HubLogCV"] = hub.HubLogStd / hub.HubLogMean.abs().clip(lower=1e-6)
    hub["HubDaysSinceLastTrade"] = (stats_end - CALENDAR_START).days - hub.HubLastTradeOrd
    hub.drop(columns=["HubVolSum", "HubSessSum", "HubLastTradeOrd"], inplace=True)

    # operating-calendar rates use the full history window, closed days included
    hist_all = panel_df[(panel_df.Date <= stats_end) & (panel_df.RowPresent == 1)]
    hub = hub.join(hist_all.groupby("HubID").agg(HubOpenRate=("IsOpen", "mean"),
                                                 HubPromoRate=("PromoActive", "mean"),
                                                 HubSchoolRate=("SchoolClosureFlag", "mean"),
                                                 HubHistDays=("IsOpen", "size")))
    hub = hub.join(hist_all[hist_all.Weekday == 7].groupby("HubID").IsOpen.mean()
                   .rename("HubSundayOpenRate"))
    hub = hub.join(hist_all[hist_all.Weekday == 6].groupby("HubID").IsOpen.mean()
                   .rename("HubSaturdayOpenRate"))
    hub["HubSundayOpenRate"] = hub.HubSundayOpenRate.fillna(0.0)
    hub["HubSaturdayOpenRate"] = hub.HubSaturdayOpenRate.fillna(0.0)
    hub_mean_map = hub.HubLogMean
    global_mean = float(stats.LogVol.mean())
    bar.update(1)

    # --------------------------------------------- 2) shrunk conditional means (+ networks)
    cond_frames = []
    for keys, name in [(["HubID", "Weekday"], "HubDowMean"),
                       (["HubID", "PromoActive"], "HubPromoMean"),
                       (["HubID", "Weekday", "PromoActive"], "HubDowPromoMean"),
                       (["HubID", "Month"], "HubMonthMean"),
                       (["HubID", "SchoolClosureFlag"], "HubSchoolMean")]:
        cond_frames.append((keys, name, _grouped_mean(stats, keys, "LogVol", hub_mean_map, name)))

    net_dow = stats.groupby("Weekday").LogVol.mean().rename("NetDowMean").reset_index()
    net_promo = stats.groupby("PromoActive").LogVol.mean().rename("NetPromoMean").reset_index()
    net_month = stats.groupby("Month").LogVol.mean().rename("NetMonthMean").reset_index()
    bar.update(1)

    # ------------------------------------------------------------- 3) trailing-window views
    trail = pd.DataFrame(index=hub.index)
    for w in CFG.TRAIL_WINDOWS:
        sub = stats[stats.Date >= stats_end - pd.Timedelta(days=w - 1)]
        g = sub.groupby("HubID").LogVol.agg(["sum", "count"]).reindex(hub.index)
        vals = _smooth(g["sum"].fillna(0.0).to_numpy(), g["count"].fillna(0.0).to_numpy(),
                       hub.HubLogMean.to_numpy())
        trail[f"HubTrailMean{w}"] = vals
        trail[f"HubTrailObs{w}"] = g["count"].fillna(0.0).to_numpy()
        trail[f"HubTrailDelta{w}"] = vals - hub.HubLogMean.to_numpy()
        if w in (91, 365):
            gd = _grouped_mean(sub, ["HubID", "Weekday"], "LogVol", hub_mean_map, f"HubTrailDow{w}")
            cond_frames.append((["HubID", "Weekday"], f"HubTrailDow{w}", gd))
        if w == 91:
            gp = _grouped_mean(sub, ["HubID", "PromoActive"], "LogVol", hub_mean_map, "HubTrailPromo91")
            cond_frames.append((["HubID", "PromoActive"], "HubTrailPromo91", gp))
    trail["HubTrend28over182"] = trail.HubTrailMean28 - trail.HubTrailMean182
    trail["HubTrend91over365"] = trail.HubTrailMean91 - trail.HubTrailMean365
    trail["HubTrend7over91"] = trail.HubTrailMean7 - trail.HubTrailMean91
    bar.update(1)

    # ------------------------------------------------------------------------ 4) assembly
    feat = target.merge(hub.join(trail).reset_index(), on="HubID", how="left")
    for keys, name, frame in cond_frames:
        feat = feat.merge(frame, on=keys, how="left")
        feat[name] = feat[name].fillna(feat.HubLogMean).fillna(global_mean)
        feat[f"{name}Delta"] = feat[name] - feat.HubLogMean
    feat = (feat.merge(net_dow, on="Weekday", how="left")
                .merge(net_promo, on="PromoActive", how="left")
                .merge(net_month, on="Month", how="left"))

    promo_pivot = _grouped_mean(stats, ["HubID", "PromoActive"], "LogVol", hub_mean_map, "_pm") \
        .pivot(index="HubID", columns="PromoActive", values="_pm")
    if {0, 1} <= set(promo_pivot.columns):
        uplift = promo_pivot[1] - promo_pivot[0]
    else:
        uplift = pd.Series(0.0, index=promo_pivot.index)
    feat["HubPromoUplift"] = feat.HubID.map(uplift).fillna(0.0)
    feat["HubVsNetwork"] = feat.HubLogMean - global_mean

    n_missing = int(feat.HubLogMean.isna().sum())
    if n_missing:
        LOG(f"{tag}: {n_missing} rows belong to hubs with no usable history -> network mean", "WARN")
        feat["HubLogMean"] = feat.HubLogMean.fillna(global_mean)

    feat["HorizonDays"] = (feat.Date - stats_end).dt.days.astype("int16")
    feat["StatsWindowDays"] = np.int16((stats_end - CALENDAR_START).days + 1)
    feat["GlobalLogMean"] = np.float32(global_mean)
    bar.update(1)
    bar.close()
    return feat


BASE_FEATURES = [
    # operating calendar (all of these are supplied in advance for the test window)
    "IsOpen", "PromoActive", "RegionalHoliday", "SchoolClosureFlag", "Weekday", "IsWeekend",
    # calendar
    "Year", "Month", "Day", "WeekOfYear", "DayOfYear", "DaysSinceStart",
    "IsMonthStart", "IsMonthMid", "IsMonthEnd", "SinDayOfYear", "CosDayOfYear",
    # hub metadata
    "HubFormat", "AssortmentTier", "CompetitorDistance", "LogCompetitorDistance",
    "CompetitorDistanceMissing", "CompetitorKnown", "CompetitorOpenMonths", "CompetitorActive",
    "LoyaltyProgram", "LoyaltyWeeks", "LoyaltyActive", "IsLoyaltyRoundMonth", "LoyaltyIntervalCount",
] + SEQ_FEATURES

HISTORY_FEATURES = [
    "HubLogMean", "HubLogStd", "HubLogMedian", "HubLogMin", "HubLogMax", "HubObs",
    "HubLogP25", "HubLogP75", "HubLogIQR", "HubLogSessMean", "HubLogSessStd",
    "HubOrdersPerSession", "HubLogCV", "HubDaysSinceLastTrade",
    "HubOpenRate", "HubPromoRate", "HubSchoolRate", "HubHistDays",
    "HubSundayOpenRate", "HubSaturdayOpenRate",
    "HubDowMean", "HubDowMeanDelta", "HubPromoMean", "HubPromoMeanDelta",
    "HubDowPromoMean", "HubDowPromoMeanDelta", "HubMonthMean", "HubMonthMeanDelta",
    "HubSchoolMean", "HubSchoolMeanDelta",
    "HubTrailDow91", "HubTrailDow91Delta", "HubTrailDow365", "HubTrailDow365Delta",
    "HubTrailPromo91", "HubTrailPromo91Delta",
    "NetDowMean", "NetPromoMean", "NetMonthMean",
    "HubPromoUplift", "HubVsNetwork", "HorizonDays", "StatsWindowDays", "GlobalLogMean",
    "HubTrend28over182", "HubTrend91over365", "HubTrend7over91",
] + [f"HubTrail{kind}{w}" for w in CFG.TRAIL_WINDOWS for kind in ("Mean", "Obs", "Delta")]

FEATURES = BASE_FEATURES + HISTORY_FEATURES
assert len(FEATURES) == len(set(FEATURES)), "duplicate feature names in FEATURES"
LOG(f"feature count: {len(FEATURES)} "
    f"({len(BASE_FEATURES)} calendar/metadata + {len(HISTORY_FEATURES)} history-derived)")

### 8.1 Rolling-origin training sets (the reason there is *no* target-encoding leak)

A hub-mean feature computed over the whole training history and then attached to rows that are
*inside* that history is a textbook target-encoding leak: each row contributes to its own
feature, so the model learns to over-trust it, and the honest validation score degrades.

This notebook avoids that entirely with a **rolling-origin** construction:

* For a block anchored at `stats_end`, training rows are taken from consecutive, non-overlapping
  42-day windows walking backwards: `(stats_end-42, stats_end]`, `(stats_end-84, stats_end-42]`,
  and so on for `CFG.N_TRAIN_ORIGINS` windows.
* Each window's features are built from **its own** statistics window, which ends the day
  before the window starts. A training row therefore never contributes to any statistic that
  describes it.
* As a bonus, training rows have `HorizonDays ∈ [1, 42]` — exactly the distribution seen at
  prediction time — so the horizon feature is learned on matched data rather than extrapolated.

* **Purpose:** materialise, for each block, the training matrix (rolling origins) and the
  target matrix (the block's own forecast window), and persist them for the GPU workers.
* **What it does:** loops the origins with a `tqdm` bar, keeps only open, positive-volume rows
  for training, attaches exponential recency weights (half-life `CFG.RECENCY_HALFLIFE_DAYS`,
  measured from the block anchor), casts all features to `float32` and writes
  `cache/<block>.parquet`.
* **Outputs:** `BLOCK_DATA`, `BLOCK_META`, `cache/*.parquet`, `metrics/block_datasets.json`.
* **Cost/benefit:** rolling origins reduce the training-row count relative to using every
  historical row, but every remaining row is described exactly as the test rows are. That
  trade is what makes the validation number trustworthy.

In [ ]:
CFG.N_TRAIN_ORIGINS = 14          # 14 x 42 days ~= 1.6 years of rolling-origin training rows


def rolling_origins(stats_end, horizon, n_origins, min_stats_days=CFG.MIN_STATS_DAYS):
    """Consecutive, non-overlapping `horizon`-day windows walking back from `stats_end`."""
    windows, cursor = [], stats_end
    for _ in range(n_origins):
        t_end = cursor
        t_start = t_end - pd.Timedelta(days=horizon - 1)
        origin = t_start - pd.Timedelta(days=1)                # statistics may use <= origin
        if (origin - CALENDAR_START).days + 1 < min_stats_days:
            break
        windows.append({"stats_end": origin, "target_start": t_start, "target_end": t_end})
        cursor = origin
    return windows


def recency_weights(dates, ref_date, halflife_days):
    """Exponential decay: a row `halflife_days` before `ref_date` receives half the weight."""
    age = (ref_date - dates).dt.days.to_numpy().astype("float64")
    return np.power(0.5, np.clip(age, 0, None) / float(halflife_days)).astype("float32")


def materialise_block(block_name):
    """Build the training (rolling-origin) and target matrices for one block."""
    blk = BLOCKS[block_name]
    anchor, t_start, t_end = blk["stats_end"], blk["target_start"], blk["target_end"]

    target = build_history_features(panel, anchor, t_start, t_end, f"{block_name}/target", progress=True)
    target["role"] = "target"

    origins = rolling_origins(anchor, CFG.VALID_DAYS, CFG.N_TRAIN_ORIGINS)
    parts = []
    for win in tqdm(origins, desc=f"train origins[{block_name}]", unit="origin", leave=False):
        part = build_history_features(panel, win["stats_end"], win["target_start"],
                                      win["target_end"], f"{block_name}/origin")
        part = part[(part.IsOpen == 1) & (part.OrderVolume > 0)]
        parts.append(part)
    train = pd.concat(parts, ignore_index=True)
    train["role"] = "train"
    assert train.Date.max() < target.Date.min(), f"{block_name}: training rows overlap the target"

    both = pd.concat([train, target], ignore_index=True)
    both["y"] = np.log1p(both.OrderVolume.to_numpy()).astype("float32")
    both["w"] = np.float32(1.0)
    m = both.role == "train"
    both.loc[m, "w"] = recency_weights(both.loc[m, "Date"], anchor, CFG.RECENCY_HALFLIFE_DAYS)

    missing = [c for c in FEATURES if c not in both.columns]
    assert not missing, f"{block_name}: feature builder did not produce {missing}"

    keep = list(dict.fromkeys(["HubID", "Date", "Id", "role", "y", "w",
                               "OrderVolume", "IsOpen", "is_test"] + FEATURES))
    out = both[keep].copy()
    out[FEATURES] = out[FEATURES].astype("float32")
    assert np.isfinite(out.loc[out.role == "train", "y"]).all(), f"{block_name}: non-finite target"

    path = OUT.cache / f"{block_name}.parquet"
    out.to_parquet(path, index=False)
    meta = {
        "block": block_name,
        "anchor_stats_end": str(anchor.date()),
        "target_range": [str(t_start.date()), str(t_end.date())],
        "train_origins": len(origins),
        "train_date_range": [str(out.loc[out.role == "train", "Date"].min().date()),
                             str(out.loc[out.role == "train", "Date"].max().date())],
        "train_rows": int(m.sum()),
        "target_rows": int((out.role == "target").sum()),
        "target_open_rows": int(((out.role == "target") & (out.IsOpen == 1)).sum()),
        "features": len(FEATURES),
        "parquet_mb": round(path.stat().st_size / 1e6, 2),
    }
    assert meta["train_date_range"][1] < meta["target_range"][0], f"{block_name}: temporal leakage"
    return out, meta


BLOCK_DATA, BLOCK_META = {}, {}
for _name in tqdm(list(BLOCKS), desc="Materialising blocks", unit="block"):
    BLOCK_DATA[_name], BLOCK_META[_name] = materialise_block(_name)
    _m = BLOCK_META[_name]
    LOG(f"block {_name:<9} origins={_m['train_origins']:>2} "
        f"train={_m['train_rows']:>7,} ({_m['train_date_range'][0]}..{_m['train_date_range'][1]}) "
        f"target={_m['target_rows']:>6,} (open {_m['target_open_rows']:>6,}) "
        f"-> {_m['parquet_mb']} MB")

(OUT.metrics / "block_datasets.json").write_text(json.dumps(BLOCK_META, indent=2), encoding="utf-8")

_final_tgt = BLOCK_DATA["final"].query("role == 'target'")
assert len(_final_tgt) == len(test_raw) and set(_final_tgt.Id) == set(test_raw.Id), \
    "the final block does not cover every test Id exactly once"
LOG("final block covers all test Ids exactly once — submission alignment guaranteed")
display(pd.DataFrame(BLOCK_META).T)

### 8.2 Leakage audit

Assertions, not prose. Three independent checks per block:

1. **Temporal ordering** — the newest training date is strictly older than the oldest target
   date.
2. **Statistics identity** — `HubLogMean` for a random sample of hubs is recomputed directly
   from `orders_train.csv` restricted to the block's statistics window and compared against the
   value stored in the feature matrix (tolerance `1e-4`, which covers `float32` storage of the feature matrix).
3. **Within-block constancy** — every hub-level history feature must take exactly one value per
   hub across the whole target window. If one varied by date, it would mean the statistic had
   absorbed information from inside the forecast window.

Also asserts that no raw target column (`OrderVolume`, `AppSessions`) appears in `FEATURES`.
A failure raises, so a leaking run can never reach the submission cell.

* **Outputs:** `metrics/leakage_audit.json`.

In [ ]:
audit = {}
rng = np.random.default_rng(CFG.SEED)

for name, blk in BLOCKS.items():
    df = BLOCK_DATA[name]
    tgt, trn = df[df.role == "target"], df[df.role == "train"]

    temporal_ok = trn.Date.max() < tgt.Date.min()

    stats_src = train_raw[(train_raw.Date <= blk["stats_end"]) & (train_raw.IsOpen == 1)
                          & (train_raw.OrderVolume > 0)]
    ref = np.log1p(stats_src.set_index("HubID").OrderVolume).groupby(level=0).mean()
    got = tgt.drop_duplicates("HubID").set_index("HubID").HubLogMean
    sample = rng.choice(np.intersect1d(ref.index.to_numpy(), got.index.to_numpy()),
                        size=min(25, len(got)), replace=False)
    max_diff = float(max(abs(float(got.loc[h]) - float(ref.loc[h])) for h in sample))
    # tolerance covers float32 storage of the feature matrix; a real leak shifts means far more

    hub_level = ["HubLogMean", "HubLogStd", "HubTrailMean28", "HubTrailMean365", "HubPromoUplift"]
    nunique = int(tgt.groupby("HubID")[hub_level].nunique().max().max())

    audit[name] = {
        "temporal_order_ok": bool(temporal_ok),
        "train_date_max": str(trn.Date.max().date()),
        "target_date_min": str(tgt.Date.min().date()),
        "hub_mean_recompute_max_abs_diff": max_diff,
        "max_distinct_hub_level_values_within_block": nunique,
        "raw_target_columns_in_FEATURES": [c for c in ("OrderVolume", "AppSessions") if c in FEATURES],
    }
    assert temporal_ok, f"{name}: training rows are not strictly before target rows"
    assert max_diff < 1e-4, f"{name}: HubLogMean disagrees with an independent recompute ({max_diff})"
    assert nunique == 1, f"{name}: a hub-level history feature varies inside the block"
    assert not audit[name]["raw_target_columns_in_FEATURES"], "a raw target column leaked into FEATURES"
    LOG(f"leakage audit {name:<9} PASS (max hub-mean recompute diff {max_diff:.2e})")

(OUT.metrics / "leakage_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
display(pd.DataFrame(audit).T)

## 9. Training configuration and the dual-T4 job runner

Kaggle's accelerator option for this notebook is **2 x NVIDIA T4**. A single XGBoost booster
cannot span two GPUs without a Dask cluster (heavyweight, and unnecessary here), so the
notebook exploits both cards the way that actually pays off for this problem size:
**independent boosters run concurrently, one per GPU**, and the seed ensemble / ablation grid
is dealt out across the two devices.

* **Purpose:** define a reusable job runner that trains a list of XGBoost jobs across all
  detected devices in parallel, with live progress.
* **What this cell does:** writes `xgb_worker.py` into the working directory. Each worker
  process:
  * is pinned to one device via `CUDA_VISIBLE_DEVICES`, so `device="cuda:0"` inside the worker
    always means *its own* card and the two processes never contend for memory;
  * memory-maps the block parquet, builds a `QuantileDMatrix` (roughly half the GPU memory of
    a dense `DMatrix`, which matters on a 16 GB T4 with ~550 k x 100 float32 rows);
  * trains with early stopping against the block's target window when one is requested;
  * emits one `PROGRESS {json}` line every 25 boosting rounds and a final `DONE {json}` line,
    which the parent turns into `tqdm` bars;
  * saves the booster (`.ubj`), the target-window predictions (`.npy`) and a metadata JSON.
* **Why subprocesses rather than threads:** XGBoost releases the GIL during training, but a
  process per GPU also isolates CUDA contexts and lets a crashed job be reported cleanly
  instead of taking down the kernel. A non-zero exit code is raised as an error — no silent
  failures.
* **CPU fallback:** with no CUDA device the same worker runs `device="cpu"` with `nthread`
  split across the available cores.

In [ ]:
WORKER_PATH = OUT.root / "xgb_worker.py"

WORKER_SOURCE = r"""
# Standalone XGBoost training worker: one process per device, jobs run sequentially.
import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd


def emit(kind, payload):
    sys.stdout.write(kind + " " + json.dumps(payload) + chr(10))
    sys.stdout.flush()


def make_callback(xgb, tag, total, every=25):
    class _CB(xgb.callback.TrainingCallback):
        def after_iteration(self, model, epoch, evals_log):
            if epoch % every == 0 or epoch == total - 1:
                metrics = {}
                for split, mdict in evals_log.items():
                    for metric, values in mdict.items():
                        metrics[split + "-" + metric] = float(values[-1])
                emit("PROGRESS", {"tag": tag, "iter": epoch + 1, "total": total, "metrics": metrics})
            return False
    return _CB()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--jobs", required=True, help="JSON file holding the list of jobs")
    ap.add_argument("--device", required=True, help="cuda:N or cpu")
    ap.add_argument("--nthread", type=int, default=0)
    args = ap.parse_args()

    if args.device.startswith("cuda"):
        # Pin this process to exactly one physical GPU; it then addresses it as cuda:0.
        os.environ["CUDA_VISIBLE_DEVICES"] = args.device.split(":")[1]
        device = "cuda:0"
    else:
        device = "cpu"

    import xgboost as xgb  # imported after CUDA_VISIBLE_DEVICES is set

    jobs = json.loads(open(args.jobs, "r", encoding="utf-8").read())
    cache = {}

    for job in jobs:
        tag = job["tag"]
        t0 = time.time()
        if job["data"] not in cache:
            cache[job["data"]] = pd.read_parquet(job["data"])
        df = cache[job["data"]]
        feats = job["features"]

        is_train = (df["role"] == "train").to_numpy()
        is_target = (df["role"] == "target").to_numpy()
        Xtr = df.loc[is_train, feats].to_numpy(dtype=np.float32)
        ytr = df.loc[is_train, "y"].to_numpy(dtype=np.float32)
        wtr = df.loc[is_train, "w"].to_numpy(dtype=np.float32) if job["use_weights"] else None

        params = dict(job["params"])
        params["device"] = device
        params["seed"] = int(job["seed"])
        if args.nthread:
            params["nthread"] = int(args.nthread)

        dtrain = xgb.QuantileDMatrix(Xtr, label=ytr, weight=wtr, feature_names=feats)
        evals = [(dtrain, "train")]
        dvalid = None
        if job["evaluate"]:
            # Only open target rows carry a usable label; closed rows are deterministic zeros.
            mask = is_target & (df["IsOpen"].to_numpy() == 1) & (df["OrderVolume"].to_numpy() > 0)
            Xva = df.loc[mask, feats].to_numpy(dtype=np.float32)
            yva = df.loc[mask, "y"].to_numpy(dtype=np.float32)
            dvalid = xgb.QuantileDMatrix(Xva, label=yva, ref=dtrain, feature_names=feats)
            evals.append((dvalid, "valid"))

        total = int(job["num_boost_round"])
        callbacks = [make_callback(xgb, tag, total, job.get("progress_every", 25))]
        kwargs = {}
        if job["evaluate"] and job.get("early_stopping"):
            kwargs["early_stopping_rounds"] = int(job["early_stopping"])

        booster = xgb.train(params, dtrain, num_boost_round=total, evals=evals,
                            verbose_eval=False, callbacks=callbacks, **kwargs)

        best_iter = int(getattr(booster, "best_iteration", total - 1))
        rng = (0, best_iter + 1)

        # predictions for every target row of this block, in the parquet's own row order
        Xall = df.loc[is_target, feats].to_numpy(dtype=np.float32)
        dall = xgb.DMatrix(Xall, feature_names=feats)
        preds = booster.predict(dall, iteration_range=rng)

        prefix = job["out_prefix"]
        booster.save_model(prefix + ".ubj")
        np.save(prefix + ".pred.npy", preds.astype(np.float32))
        score_map = booster.get_score(importance_type="gain")
        weight_map = booster.get_score(importance_type="weight")
        meta = {
            "tag": tag,
            "device": args.device,
            "seed": int(job["seed"]),
            "best_iteration": best_iter,
            "best_score": float(getattr(booster, "best_score", float("nan"))),
            "num_boost_round": total,
            "train_rows": int(is_train.sum()),
            "target_rows": int(is_target.sum()),
            "use_weights": bool(job["use_weights"]),
            "seconds": round(time.time() - t0, 1),
            "importance_gain": {k: float(v) for k, v in score_map.items()},
            "importance_weight": {k: float(v) for k, v in weight_map.items()},
            "params": {k: v for k, v in params.items()},
        }
        with open(prefix + ".meta.json", "w", encoding="utf-8") as fh:
            json.dump(meta, fh, indent=2)
        emit("DONE", {"tag": tag, "best_iteration": best_iter,
                      "best_score": meta["best_score"], "seconds": meta["seconds"]})

    emit("WORKER_DONE", {"device": args.device, "jobs": len(jobs)})


if __name__ == "__main__":
    main()
"""

WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
LOG(f"worker written -> {WORKER_PATH} ({WORKER_PATH.stat().st_size / 1024:.1f} KB)")

### 9.1 The parent-side dispatcher

* **Purpose:** deal a list of job specifications across the available devices, run them
  concurrently and surface live training progress.
* **What it does:** round-robins jobs onto devices, writes one job file per device, launches
  one worker subprocess per device, and reads each worker's stdout on a background thread.
  `PROGRESS` lines drive a per-device `tqdm` bar whose postfix shows the current job, boosting
  iteration, train RMSE and validation RMSE (which *is* RMSLE, since the target is `log1p`);
  `DONE` lines advance a global job bar. A second thread drains each worker's stderr into
  `logs/worker_<device>.err.log` — mandatory, because the stderr pipe holds only ~64 KB and a
  worker that filled it while the parent read only stdout would hang forever — and its tail is
  echoed if the worker fails.
* **Inputs:** a list of job dicts. **Outputs:** a dict `tag -> {meta, preds}` loaded back from
  the artefacts the workers wrote, and `logs/worker_<device>.log`.
* **Error handling:** any non-zero worker exit raises `RuntimeError` with the captured stderr;
  a missing artefact for an expected tag also raises. Nothing downstream can proceed on
  partial results.

In [ ]:
import threading
import queue as _queue
from collections import deque


def _read_stdout(stream, sink, q, device):
    """Stream a worker's stdout into the shared queue, and tee it to a log file."""
    with open(sink, "w", encoding="utf-8") as fh:
        for line in iter(stream.readline, ""):
            fh.write(line)
            line = line.strip()
            if line.startswith(("PROGRESS ", "DONE ", "WORKER_DONE ")):
                kind, _, payload = line.partition(" ")
                try:
                    q.put((device, kind, json.loads(payload)))
                except json.JSONDecodeError:
                    pass
    stream.close()


def _read_stderr(stream, sink, tail):
    """Drain a worker's stderr continuously.

    This is not optional: the stderr pipe holds only ~64 KB, so a worker that writes more than
    that (XGBoost emits warnings there) would block forever if the parent read only stdout.
    The last lines are retained so a failure can be reported with its real cause.
    """
    with open(sink, "w", encoding="utf-8") as fh:
        for line in iter(stream.readline, ""):
            fh.write(line)
            tail.append(line.rstrip())
    stream.close()


def run_jobs(jobs, devices=None, desc="training"):
    """Run XGBoost jobs across every device in parallel; return {tag: {meta, preds}}."""
    devices = devices or DEVICES
    if not jobs:
        return {}
    per_device = {d: [] for d in devices}
    for i, job in enumerate(jobs):
        per_device[devices[i % len(devices)]].append(job)

    nthread = max(1, mp.cpu_count() // max(1, len(devices)))
    procs, q = {}, _queue.Queue()
    threads, stderr_tails = [], {}
    for dev, dev_jobs in per_device.items():
        if not dev_jobs:
            continue
        job_file = OUT.cache / f"jobs_{dev.replace(':', '')}_{int(time.time() * 1000) % 10**7}.json"
        job_file.write_text(json.dumps(dev_jobs), encoding="utf-8")
        proc = subprocess.Popen(
            [sys.executable, str(WORKER_PATH), "--jobs", str(job_file),
             "--device", dev, "--nthread", str(nthread)],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1,
            cwd=str(OUT.root))
        procs[dev] = proc
        stderr_tails[dev] = deque(maxlen=60)
        safe = dev.replace(":", "")
        for target, targs in (
            (_read_stdout, (proc.stdout, OUT.logs / f"worker_{safe}.log", q, dev)),
            (_read_stderr, (proc.stderr, OUT.logs / f"worker_{safe}.err.log", stderr_tails[dev])),
        ):
            t = threading.Thread(target=target, args=targs, daemon=True)
            t.start()
            threads.append(t)
        LOG(f"launched worker on {dev} with {len(dev_jobs)} job(s) (nthread={nthread})")

    job_bar = tqdm(total=len(jobs), desc=f"{desc}: jobs", unit="model", position=0)
    dev_bars = {d: tqdm(total=0, desc=f"  {d}", position=i + 1, bar_format="{desc} {postfix}")
                for i, d in enumerate(procs)}
    iter_bars = {}
    finished = 0
    t_start = time.time()
    try:
        while finished < len(procs):
            try:
                dev, kind, payload = q.get(timeout=CFG.WORKER_POLL_SEC)
            except _queue.Empty:
                for dev, proc in procs.items():
                    if proc.poll() is not None and proc.returncode != 0:
                        raise RuntimeError(f"worker on {dev} exited with code {proc.returncode}:" + chr(10)
                                           + (chr(10).join(stderr_tails[dev]) or "<no stderr>"))
                if all(p.poll() is not None for p in procs.values()) and q.empty():
                    break   # every worker exited cleanly and nothing is left to read
                continue
            if kind == "PROGRESS":
                tag = payload["tag"]
                if tag not in iter_bars:
                    iter_bars[tag] = tqdm(total=payload["total"], desc=f"    {tag}",
                                          unit="iter", leave=False)
                bar = iter_bars[tag]
                bar.n = payload["iter"]
                m = payload["metrics"]
                bar.set_postfix({k: f"{v:.5f}" for k, v in m.items()}, refresh=True)
                dev_bars[dev].set_postfix_str(
                    f"{tag} it={payload['iter']}/{payload['total']} "
                    + " ".join(f"{k}={v:.5f}" for k, v in m.items()))
            elif kind == "DONE":
                tag = payload["tag"]
                if tag in iter_bars:
                    iter_bars[tag].close()
                    iter_bars.pop(tag)
                job_bar.update(1)
                LOG(f"  finished {tag}: best_iter={payload['best_iteration']} "
                    f"best_rmsle={payload['best_score']:.5f} in {payload['seconds']}s")
            elif kind == "WORKER_DONE":
                finished += 1
    finally:
        for bar in list(iter_bars.values()) + list(dev_bars.values()) + [job_bar]:
            bar.close()
        for t in threads:
            t.join(timeout=10)
        for dev, proc in procs.items():
            proc.wait()
            if proc.returncode != 0:
                err = chr(10).join(stderr_tails.get(dev, [])) or "<no stderr>"
                raise RuntimeError(f"worker on {dev} failed (exit {proc.returncode}):" + chr(10) + err)

    LOG(f"{desc}: {len(jobs)} job(s) across {len(procs)} device(s) in {time.time() - t_start:.0f}s")

    results = {}
    for job in jobs:
        prefix = Path(job["out_prefix"])
        meta_path, pred_path = Path(str(prefix) + ".meta.json"), Path(str(prefix) + ".pred.npy")
        if not (meta_path.exists() and pred_path.exists()):
            raise RuntimeError(f"worker produced no artefacts for job {job['tag']}")
        results[job["tag"]] = {"meta": json.loads(meta_path.read_text(encoding="utf-8")),
                               "preds": np.load(pred_path)}
    return results


def make_job(tag, block, seed, *, evaluate, num_boost_round, use_weights=True,
             params=None, early_stopping=None, progress_every=25):
    """Build one job specification for the worker."""
    return {
        "tag": tag,
        "data": str(OUT.cache / f"{block}.parquet"),
        "features": FEATURES,
        "params": {**CFG.XGB_PARAMS, **(params or {})},
        "num_boost_round": int(num_boost_round),
        "early_stopping": early_stopping,
        "evaluate": bool(evaluate),
        "use_weights": bool(use_weights),
        "seed": int(seed),
        "out_prefix": str(OUT.models / tag),
        "progress_every": progress_every,
    }


LOG(f"dispatcher ready — devices: {DEVICES}")

## 10. Metrics and baselines

* **Purpose:** implement RMSLE exactly as the competition defines it, and fix reference points
  that any model must beat before it is worth trusting.
* **What it does:**
  * `rmsle(actual, predicted)` works on the raw scale; `rmsle_from_log` avoids a redundant
    `expm1`/`log1p` round-trip when both sides are already in log space.
  * Builds four naive baselines on the validation block: global mean, hub mean,
    hub x weekday mean, and hub x weekday x promo mean (all computed from the *statistics
    window only*, so they are honest forecasts rather than fitted values).
* **Scoring convention (applies everywhere below):** predictions for rows with `IsOpen == 0`
  are forced to 0. In the training history `IsOpen == 0` implies `OrderVolume == 0` without
  exception, so this is a fact about the data rather than an assumption, and it removes ~17% of
  rows from any modelling error.
* **Outputs:** `metrics/baselines.json`, `VALID_EVAL` (the evaluation frame reused by every
  later section).

In [ ]:
def rmsle(actual, predicted):
    """Competition metric on the raw order scale."""
    a = np.asarray(actual, dtype="float64")
    p = np.clip(np.asarray(predicted, dtype="float64"), 0, None)
    return float(np.sqrt(np.mean((np.log1p(p) - np.log1p(a)) ** 2)))


def rmsle_from_log(log_actual, log_pred):
    """RMSLE when both arguments are already log1p-transformed."""
    return float(np.sqrt(np.mean((np.asarray(log_pred, dtype="float64")
                                  - np.asarray(log_actual, dtype="float64")) ** 2)))


def postprocess(log_preds, is_open, multiplier=1.0):
    """log-space predictions -> raw orders, closed hubs zeroed, non-negative, rounded."""
    raw = np.expm1(np.asarray(log_preds, dtype="float64")) * float(multiplier)
    raw = np.clip(raw, 0.0, None)
    raw = np.where(np.asarray(is_open) == 1, raw, 0.0)
    return np.rint(raw)


# ---- evaluation frame for the validation block -------------------------------------------
VALID_EVAL = BLOCK_DATA["valid"].query("role == 'target'").reset_index(drop=True)
VALID_EVAL_BT = BLOCK_DATA["backtest"].query("role == 'target'").reset_index(drop=True)
LOG(f"validation rows {len(VALID_EVAL):,} (open {int((VALID_EVAL.IsOpen == 1).sum()):,}); "
    f"backtest rows {len(VALID_EVAL_BT):,}")

# ---- naive baselines, built from the statistics window only --------------------------------
def baseline_predictions(eval_df, stats_end):
    src = train_raw[(train_raw.Date <= stats_end) & (train_raw.IsOpen == 1) & (train_raw.OrderVolume > 0)].copy()
    src["LogVol"] = np.log1p(src.OrderVolume)
    g_mean = src.LogVol.mean()
    hub_mean = src.groupby("HubID").LogVol.mean()
    dow_mean = src.groupby(["HubID", "Weekday"]).LogVol.mean()
    dow_promo = src.groupby(["HubID", "Weekday", "PromoActive"]).LogVol.mean()

    idx_hub = eval_df.HubID.to_numpy()
    out = {}
    out["global_mean"] = np.full(len(eval_df), g_mean)
    out["hub_mean"] = pd.Series(idx_hub).map(hub_mean).fillna(g_mean).to_numpy()
    keys_dow = list(zip(eval_df.HubID, eval_df.Weekday.astype(int)))
    out["hub_weekday_mean"] = pd.Series(keys_dow).map(dow_mean).fillna(
        pd.Series(idx_hub).map(hub_mean)).fillna(g_mean).to_numpy()
    keys_dp = list(zip(eval_df.HubID, eval_df.Weekday.astype(int), eval_df.PromoActive.astype(int)))
    out["hub_weekday_promo_mean"] = pd.Series(keys_dp).map(dow_promo).fillna(
        pd.Series(keys_dow).map(dow_mean)).fillna(pd.Series(idx_hub).map(hub_mean)).fillna(g_mean).to_numpy()
    return out


BASELINES = {}
for _blk_name, _eval_df in [("valid", VALID_EVAL), ("backtest", VALID_EVAL_BT)]:
    _stats_end = BLOCKS[_blk_name]["stats_end"]
    _actual = _eval_df.OrderVolume.to_numpy()
    _open = _eval_df.IsOpen.to_numpy()
    for _bname, _logp in tqdm(baseline_predictions(_eval_df, _stats_end).items(),
                              desc=f"baselines[{_blk_name}]", unit="baseline"):
        BASELINES[f"{_blk_name}:{_bname}"] = rmsle(_actual, postprocess(_logp, _open))

(OUT.metrics / "baselines.json").write_text(json.dumps(BASELINES, indent=2), encoding="utf-8")
for _k, _v in BASELINES.items():
    LOG(f"baseline RMSLE  {_k:<34} {_v:.5f}")

## 11. Stage A — sample-weighting ablation and boosting-round selection (back-test block)

The number of boosting rounds and the choice of recency weighting are *hyper-parameters*. If
they were chosen by early stopping on the validation block, the validation RMSLE would no
longer be an unbiased estimate of leaderboard performance. Stage A therefore does all of that
tuning on the **earlier back-test block**, which the validation block never touches.

* **Purpose:** decide (a) whether exponential recency weighting helps, and (b) how many
  boosting rounds this feature set needs.
* **What it does:** launches two jobs — weighted and unweighted — on the `backtest` block with
  early stopping against that block's 42-day target window. Both run concurrently, one per T4.
  Live `tqdm` bars show iteration, train RMSE and validation RMSE (= RMSLE in log space).
* **Inputs:** `cache/backtest.parquet`. **Outputs:** `STAGE_A` results, `BEST_WEIGHTING`,
  `BEST_ROUNDS`, `metrics/stage_a_ablation.json`, booster/metadata artefacts in `models/`.
* **Design note:** early stopping needs a scored window, and `backtest` is the only block we
  are willing to "spend" on tuning. The validation block stays clean for Section 12.

In [ ]:
stage_a_jobs = [
    make_job(tag="A_backtest_weighted", block="backtest", seed=CFG.SEED,
             evaluate=True, num_boost_round=CFG.NUM_BOOST_ROUND,
             use_weights=True, early_stopping=CFG.EARLY_STOPPING),
    make_job(tag="A_backtest_unweighted", block="backtest", seed=CFG.SEED,
             evaluate=True, num_boost_round=CFG.NUM_BOOST_ROUND,
             use_weights=False, early_stopping=CFG.EARLY_STOPPING),
]

STAGE_A = run_jobs(stage_a_jobs, desc="stage A (ablation)")

stage_a_rows = []
for tag, res in STAGE_A.items():
    meta = res["meta"]
    eval_df = VALID_EVAL_BT
    preds = postprocess(res["preds"], eval_df.IsOpen.to_numpy())
    stage_a_rows.append({
        "tag": tag,
        "use_weights": meta["use_weights"],
        "best_iteration": meta["best_iteration"],
        "early_stop_rmsle_open_rows": meta["best_score"],
        "backtest_rmsle_all_rows": rmsle(eval_df.OrderVolume.to_numpy(), preds),
        "seconds": meta["seconds"],
        "device": meta["device"],
    })
stage_a_df = pd.DataFrame(stage_a_rows).sort_values("backtest_rmsle_all_rows").reset_index(drop=True)

best = stage_a_df.iloc[0]
BEST_WEIGHTING = bool(best.use_weights)
BEST_ROUNDS = int(max(200, best.best_iteration + 1))
STAGE_A_SUMMARY = {
    "results": stage_a_rows,
    "chosen_use_weights": BEST_WEIGHTING,
    "chosen_rounds": BEST_ROUNDS,
    "recency_halflife_days": CFG.RECENCY_HALFLIFE_DAYS,
    "selection_block": "backtest (validation block untouched)",
}
(OUT.metrics / "stage_a_ablation.json").write_text(json.dumps(STAGE_A_SUMMARY, indent=2), encoding="utf-8")

LOG(f"stage A winner: use_weights={BEST_WEIGHTING}, rounds={BEST_ROUNDS}, "
    f"backtest RMSLE={best.backtest_rmsle_all_rows:.5f}")
display(stage_a_df)

## 12. Stage B — honest validation with a fixed round budget

* **Purpose:** produce an unbiased estimate of leaderboard RMSLE, and measure how much the
  seed ensemble actually buys.
* **What it does:** trains `CFG.N_SEEDS` boosters on the `valid` block with the Stage-A
  configuration and a **fixed** `BEST_ROUNDS` budget — no early stopping, so the validation
  window is never used to make a decision before it is scored. Jobs alternate across the two
  T4s. Predictions are averaged **in log space**, which is the right pooling for a squared-log
  metric (it is the arithmetic mean of the quantity the metric squares).
* **Inputs:** `cache/valid.parquet`. **Outputs:** `STAGE_B`, `VALID_LOG_PRED` (ensemble mean),
  per-seed and ensemble RMSLE in `metrics/stage_b_validation.json`.
* **Note:** `evaluate=True` is passed only so the live bars can show the validation curve; with
  `early_stopping=None` it cannot influence the model that gets scored.

In [ ]:
stage_b_jobs = [
    make_job(tag=f"B_valid_seed{CFG.SEED + i}", block="valid", seed=CFG.SEED + i,
             evaluate=True, num_boost_round=BEST_ROUNDS,
             use_weights=BEST_WEIGHTING, early_stopping=None)
    for i in range(CFG.N_SEEDS)
]

STAGE_B = run_jobs(stage_b_jobs, desc="stage B (validation ensemble)")

valid_actual = VALID_EVAL.OrderVolume.to_numpy()
valid_open = VALID_EVAL.IsOpen.to_numpy()

per_seed, running = [], []
for i, (tag, res) in enumerate(sorted(STAGE_B.items()), start=1):
    log_pred = res["preds"]
    running.append(log_pred)
    per_seed.append({
        "tag": tag,
        "seed": res["meta"]["seed"],
        "device": res["meta"]["device"],
        "seconds": res["meta"]["seconds"],
        "rmsle_single": rmsle(valid_actual, postprocess(log_pred, valid_open)),
        "rmsle_running_ensemble": rmsle(valid_actual, postprocess(np.mean(running, axis=0), valid_open)),
        "n_models_in_ensemble": i,
    })

VALID_LOG_PRED = np.mean([res["preds"] for res in STAGE_B.values()], axis=0)
VALID_RAW_PRED = postprocess(VALID_LOG_PRED, valid_open)
VALID_RMSLE = rmsle(valid_actual, VALID_RAW_PRED)

seed_df = pd.DataFrame(per_seed)
STAGE_B_SUMMARY = {
    "rounds": BEST_ROUNDS,
    "use_weights": BEST_WEIGHTING,
    "per_seed": per_seed,
    "ensemble_rmsle": VALID_RMSLE,
    "mean_single_model_rmsle": float(seed_df.rmsle_single.mean()),
    "ensemble_gain": float(seed_df.rmsle_single.mean() - VALID_RMSLE),
    "best_baseline_rmsle": min(v for k, v in BASELINES.items() if k.startswith("valid:")),
}
(OUT.metrics / "stage_b_validation.json").write_text(json.dumps(STAGE_B_SUMMARY, indent=2), encoding="utf-8")

LOG(f"validation RMSLE  single-model mean : {STAGE_B_SUMMARY['mean_single_model_rmsle']:.5f}")
LOG(f"validation RMSLE  {CFG.N_SEEDS}-seed ensemble  : {VALID_RMSLE:.5f} "
    f"(gain {STAGE_B_SUMMARY['ensemble_gain']:+.5f})")
LOG(f"best naive baseline on the same block: {STAGE_B_SUMMARY['best_baseline_rmsle']:.5f} "
    f"-> improvement {STAGE_B_SUMMARY['best_baseline_rmsle'] - VALID_RMSLE:+.5f}")
display(seed_df)

### 12.1 Global multiplicative calibration

Forecasts trained on a squared-error objective in log space are, after `expm1`, slightly biased
low relative to the conditional mean; trailing-window features also lag a mildly upward network
trend. A single scalar multiplier applied to raw predictions is the standard, low-variance fix.

* **Purpose:** fit **one** parameter — a global multiplier — on the validation block and check
  that it transfers to the independent back-test block rather than merely fitting noise.
* **What it does:** sweeps `CFG.CALIB_GRID` (0.940 → 1.110) on the validation ensemble
  predictions, records the RMSLE curve, and repeats the sweep on the Stage-A back-test
  predictions. The chosen multiplier is accepted only if the two optima are within 0.02 of each
  other and the gain exceeds 1e-4; otherwise the notebook falls back to 1.0 and says so.
* **Outputs:** `CALIB_MULTIPLIER`, `metrics/calibration.json`, `figures/calibration_curve.png`.
* **Risk acknowledged:** one parameter fitted on the validation block makes the reported
  validation RMSLE very slightly optimistic. The back-test agreement check is what keeps this
  honest, and the effect of a single scalar is far smaller than the gain it delivers.

In [ ]:
def calibration_curve(actual, log_pred, is_open, grid=CFG.CALIB_GRID):
    return np.array([rmsle(actual, postprocess(log_pred, is_open, m)) for m in grid])


valid_curve = calibration_curve(valid_actual, VALID_LOG_PRED, valid_open)
best_valid_m = float(CFG.CALIB_GRID[int(np.argmin(valid_curve))])

bt_log_pred = STAGE_A[stage_a_df.iloc[0].tag]["preds"]
bt_actual = VALID_EVAL_BT.OrderVolume.to_numpy()
bt_open = VALID_EVAL_BT.IsOpen.to_numpy()
bt_curve = calibration_curve(bt_actual, bt_log_pred, bt_open)
best_bt_m = float(CFG.CALIB_GRID[int(np.argmin(bt_curve))])

gain = float(VALID_RMSLE - valid_curve.min())
for _name, _curve, _m in (("validation", valid_curve, best_valid_m), ("back-test", bt_curve, best_bt_m)):
    if int(np.argmin(_curve)) in (0, len(_curve) - 1):
        LOG(f"{_name} calibration optimum sits on a grid boundary ({_m}) — widen CFG.CALIB_GRID "
            f"if this is not a fluke", "WARN")

agree = abs(best_valid_m - best_bt_m) <= 0.02
CALIB_MULTIPLIER = best_valid_m if (agree and gain > 1e-4) else 1.0

CALIB = {
    "grid": CFG.CALIB_GRID,
    "valid_curve": [float(v) for v in valid_curve],
    "backtest_curve": [float(v) for v in bt_curve],
    "best_multiplier_valid": best_valid_m,
    "best_multiplier_backtest": best_bt_m,
    "blocks_agree_within_0.02": bool(agree),
    "valid_gain_from_calibration": gain,
    "applied_multiplier": CALIB_MULTIPLIER,
    "rmsle_valid_uncalibrated": VALID_RMSLE,
    "rmsle_valid_calibrated": rmsle(valid_actual, postprocess(VALID_LOG_PRED, valid_open, CALIB_MULTIPLIER)),
}
(OUT.metrics / "calibration.json").write_text(json.dumps(CALIB, indent=2), encoding="utf-8")

VALID_RAW_PRED = postprocess(VALID_LOG_PRED, valid_open, CALIB_MULTIPLIER)
VALID_RMSLE_CAL = CALIB["rmsle_valid_calibrated"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(CFG.CALIB_GRID, valid_curve, label="validation block", color="#c05621")
ax.plot(CFG.CALIB_GRID, bt_curve, label="back-test block", color="#805ad5")
ax.axvline(best_valid_m, ls="--", lw=1, color="#c05621")
ax.axvline(best_bt_m, ls="--", lw=1, color="#805ad5")
ax.axvline(CALIB_MULTIPLIER, color="#2f855a", lw=2, alpha=0.6, label=f"applied = {CALIB_MULTIPLIER}")
ax.set_xlabel("multiplier applied to raw predictions")
ax.set_ylabel("RMSLE")
ax.set_title("Global calibration sweep (optima agree across two independent future blocks)")
ax.legend()
save_fig(fig, "calibration_curve")

LOG(f"calibration: valid optimum {best_valid_m}, backtest optimum {best_bt_m}, "
    f"agreement={agree} -> applying {CALIB_MULTIPLIER}")
LOG(f"validation RMSLE calibrated: {VALID_RMSLE_CAL:.5f} (was {VALID_RMSLE:.5f})")

## 13. Validation evaluation — aggregate, per-segment and per-horizon metrics

* **Purpose:** go beyond a single number and show *where* the model is accurate and where it is
  not, which is what an operations team actually needs.
* **What it does:** computes RMSLE plus supporting metrics (MAE, RMSE, MAPE on non-zero
  actuals, bias, R² in log space) overall and broken down by: hub format, assortment tier,
  promo state, weekday, school-closure flag, forecast-horizon week, hub size decile, and
  renovation-affected vs normal hubs. Per-hub RMSLE is also computed and saved in full.
* **Inputs:** `VALID_EVAL`, `VALID_RAW_PRED`. **Outputs:**
  `metrics/validation_overall.json`, `metrics/validation_segments.csv`,
  `metrics/validation_per_hub.csv`.
* **Convention:** segment metrics are computed over *all* rows of the segment, including the
  closed days that are predicted as exact zeros, because that is how the leaderboard scores.

In [ ]:
EVAL = VALID_EVAL.copy()
EVAL["pred"] = VALID_RAW_PRED
EVAL["actual"] = valid_actual
EVAL["log_pred"] = np.log1p(EVAL.pred)
EVAL["log_actual"] = np.log1p(EVAL.actual)
EVAL["log_error"] = EVAL.log_pred - EVAL.log_actual
EVAL["abs_log_error"] = EVAL.log_error.abs()
EVAL["HorizonWeek"] = ((EVAL.HorizonDays - 1) // 7 + 1).astype(int)
EVAL = EVAL.merge(hub_meta_clean[["HubID", "HubFormat", "AssortmentTier"]]
                  .rename(columns={"HubFormat": "HubFormatCat", "AssortmentTier": "AssortmentCat"}),
                  on="HubID", how="left")
EVAL["Renovated"] = EVAL.HubID.isin(RENOVATION_HUBS).astype(int)
_hub_size = open_pos.groupby("HubID").OrderVolume.mean()
EVAL["HubSizeDecile"] = pd.qcut(EVAL.HubID.map(_hub_size), 10, labels=False, duplicates="drop")


def metric_block(df):
    a, p = df.actual.to_numpy(), df.pred.to_numpy()
    la, lp = np.log1p(a), np.log1p(p)
    nz = a > 0
    ss_res = float(((lp - la) ** 2).sum())
    ss_tot = float(((la - la.mean()) ** 2).sum()) if len(la) > 1 else 0.0
    return pd.Series({
        "n": int(len(df)),
        "rmsle": float(np.sqrt(np.mean((lp - la) ** 2))),
        "mae": float(np.mean(np.abs(p - a))),
        "rmse": float(np.sqrt(np.mean((p - a) ** 2))),
        "mape_nonzero_pct": float(np.mean(np.abs(p[nz] - a[nz]) / a[nz]) * 100) if nz.any() else np.nan,
        "log_bias": float(np.mean(lp - la)),
        "median_log_error": float(np.median(lp - la)),
        "r2_log": float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
        "mean_actual": float(a.mean()),
        "mean_pred": float(p.mean()),
    })


OVERALL = metric_block(EVAL).to_dict()
OVERALL_OPEN = metric_block(EVAL[EVAL.IsOpen == 1]).to_dict()
VALIDATION_OVERALL = {
    "all_rows": OVERALL,
    "open_rows_only": OVERALL_OPEN,
    "calibration_multiplier": CALIB_MULTIPLIER,
    "rounds": BEST_ROUNDS,
    "n_models": CFG.N_SEEDS,
    "use_weights": BEST_WEIGHTING,
    "baselines": {k: v for k, v in BASELINES.items() if k.startswith("valid:")},
}
(OUT.metrics / "validation_overall.json").write_text(json.dumps(VALIDATION_OVERALL, indent=2), encoding="utf-8")

segments = {
    "HubFormat": "HubFormatCat", "AssortmentTier": "AssortmentCat", "PromoActive": "PromoActive",
    "Weekday": "Weekday", "SchoolClosureFlag": "SchoolClosureFlag", "HorizonWeek": "HorizonWeek",
    "HubSizeDecile": "HubSizeDecile", "RenovationHub": "Renovated", "IsOpen": "IsOpen",
}
seg_rows = []
for label, col in tqdm(segments.items(), desc="segment metrics", unit="segment"):
    for value, grp in EVAL.groupby(col, observed=True):
        row = metric_block(grp).to_dict()
        row.update({"segment": label, "value": value})
        seg_rows.append(row)
SEGMENT_METRICS = pd.DataFrame(seg_rows)[
    ["segment", "value", "n", "rmsle", "mae", "rmse", "mape_nonzero_pct",
     "log_bias", "median_log_error", "r2_log", "mean_actual", "mean_pred"]]
SEGMENT_METRICS.to_csv(OUT.metrics / "validation_segments.csv", index=False)

EVAL["sq_log_error"] = EVAL.log_error ** 2
EVAL["abs_error"] = (EVAL.pred - EVAL.actual).abs()
_ph = EVAL.groupby("HubID").agg(n=("actual", "size"), mean_sq_log_error=("sq_log_error", "mean"),
                                mae=("abs_error", "mean"), log_bias=("log_error", "mean"),
                                mean_actual=("actual", "mean"), mean_pred=("pred", "mean"),
                                closed_days=("IsOpen", lambda s: int((s == 0).sum())))
_ph["rmsle"] = np.sqrt(_ph.mean_sq_log_error)
PER_HUB = _ph.drop(columns=["mean_sq_log_error"]).reset_index().sort_values("rmsle", ascending=False)
PER_HUB.to_csv(OUT.metrics / "validation_per_hub.csv", index=False)

LOG(f"validation RMSLE (all rows)   : {OVERALL['rmsle']:.5f}")
LOG(f"validation RMSLE (open rows)  : {OVERALL_OPEN['rmsle']:.5f}")
LOG(f"validation log-space bias     : {OVERALL['log_bias']:+.5f}")
LOG(f"validation MAE / MAPE         : {OVERALL['mae']:.1f} orders / {OVERALL['mape_nonzero_pct']:.2f}%")
LOG(f"per-hub RMSLE  p50={PER_HUB.rmsle.median():.4f}  p90={PER_HUB.rmsle.quantile(0.9):.4f}  "
    f"worst={PER_HUB.rmsle.max():.4f}")
display(SEGMENT_METRICS.head(25))

## 14. Visualisation and diagnostics

* **Purpose:** turn the validation block into the diagnostic plots a reviewer needs to trust the
  model.
* **Panels produced:**
  1. **Predicted vs actual** (log-log hexbin) with the 45° line — shows calibration across four
     orders of magnitude.
  2. **Residual distribution** in log space with the mean/median marked — should be centred and
     roughly symmetric if the metric-aligned objective is working.
  3. **RMSLE by forecast horizon week** — quantifies how quickly accuracy decays over the
     42-day window, the single most important operational number.
  4. **Network daily actual vs predicted** over the validation window — reveals systematic
     weekday or holiday misses.
  5. **Residual vs predicted level** — checks for heteroscedastic bias between small and large
     hubs (the exact failure mode RMSLE is meant to expose).
  6. **Per-hub RMSLE distribution** with the worst decile highlighted.
* **Inputs:** `EVAL`, `PER_HUB`. **Outputs:** `figures/diagnostics_*.png`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

open_eval = EVAL[EVAL.IsOpen == 1]
hb = axes[0, 0].hexbin(open_eval.log_actual, open_eval.log_pred, gridsize=60,
                       bins="log", cmap="Blues", mincnt=1)
lims = [open_eval.log_actual.min(), open_eval.log_actual.max()]
axes[0, 0].plot(lims, lims, color="#c05621", lw=1.5)
axes[0, 0].set_xlabel("log1p(actual)"); axes[0, 0].set_ylabel("log1p(predicted)")
axes[0, 0].set_title("Predicted vs actual (open rows)")
fig.colorbar(hb, ax=axes[0, 0], label="log10(count)")

axes[0, 1].hist(open_eval.log_error, bins=120, color="#2b6cb0")
axes[0, 1].axvline(0, color="#1a202c", lw=1)
axes[0, 1].axvline(open_eval.log_error.mean(), color="#c05621", lw=1.5,
                   label=f"mean {open_eval.log_error.mean():+.4f}")
axes[0, 1].axvline(open_eval.log_error.median(), color="#2f855a", lw=1.5, ls="--",
                   label=f"median {open_eval.log_error.median():+.4f}")
axes[0, 1].set_title("Log-space residuals (open rows)")
axes[0, 1].set_xlabel("log1p(pred) - log1p(actual)")
axes[0, 1].legend(fontsize=8)

hz = SEGMENT_METRICS.query("segment == 'HorizonWeek'").sort_values("value")
axes[0, 2].bar(hz.value.astype(int), hz.rmsle, color="#805ad5")
axes[0, 2].set_title("RMSLE by forecast-horizon week")
axes[0, 2].set_xlabel("weeks ahead of the last observed day")
for x, y in zip(hz.value.astype(int), hz.rmsle):
    axes[0, 2].text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=8)

daily = EVAL.groupby("Date")[["actual", "pred"]].sum() / 1e6
axes[1, 0].plot(daily.index, daily.actual, label="actual", color="#1a202c", lw=1.5)
axes[1, 0].plot(daily.index, daily.pred, label="predicted", color="#c05621", lw=1.5, ls="--")
axes[1, 0].set_title("Network daily volume, validation window")
axes[1, 0].set_ylabel("millions of orders")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].legend()

bins = pd.qcut(open_eval.log_pred, 20, duplicates="drop")
grp = open_eval.groupby(bins, observed=True).log_error.agg(["mean", "std", "size"])
centres = [float(iv.mid) for iv in grp.index]
axes[1, 1].errorbar(centres, grp["mean"], yerr=grp["std"], fmt="o-", color="#2b6cb0", capsize=3)
axes[1, 1].axhline(0, color="#c05621", lw=1)
axes[1, 1].set_title("Residual vs predicted level (mean ± sd)")
axes[1, 1].set_xlabel("log1p(predicted)"); axes[1, 1].set_ylabel("log error")

axes[1, 2].hist(PER_HUB.rmsle, bins=60, color="#2b6cb0")
_p90 = PER_HUB.rmsle.quantile(0.9)
axes[1, 2].axvline(_p90, color="#c05621", lw=1.5, label=f"p90 = {_p90:.3f}")
axes[1, 2].axvline(PER_HUB.rmsle.median(), color="#2f855a", lw=1.5, ls="--",
                   label=f"median = {PER_HUB.rmsle.median():.3f}")
axes[1, 2].set_title("Per-hub RMSLE distribution")
axes[1, 2].legend(fontsize=8)

save_fig(fig, "diagnostics_validation")

### 14.1 Feature importance and model agreement

* **Purpose:** show what the model actually uses and confirm the ensemble members agree.
* **What it does:** averages gain-based importance across the Stage-B boosters (gain is the
  right choice here — split counts over-reward high-cardinality numeric features), groups it by
  feature family, and measures the pairwise correlation and standard deviation of the seeds'
  log-space predictions.
* **Inputs:** `STAGE_B` metadata. **Outputs:** `metrics/feature_importance.csv`,
  `metrics/model_agreement.json`, `figures/feature_importance.png`.
* **Interpretation to expect:** hub-level and trailing-window level features should dominate
  (they carry the hub's scale), followed by promo and weekday shape features. If a pure calendar
  feature such as `DaysSinceStart` ranked first it would be a warning sign of trend
  extrapolation, which trees cannot do.

In [ ]:
imp = {}
for tag, res in STAGE_B.items():
    for feat, gain in res["meta"]["importance_gain"].items():
        imp.setdefault(feat, []).append(gain)
IMPORTANCE = (pd.DataFrame({"feature": list(imp), "gain_mean": [np.mean(v) for v in imp.values()],
                            "gain_std": [np.std(v) for v in imp.values()],
                            "n_models": [len(v) for v in imp.values()]})
              .sort_values("gain_mean", ascending=False).reset_index(drop=True))
IMPORTANCE["gain_share_pct"] = 100 * IMPORTANCE.gain_mean / IMPORTANCE.gain_mean.sum()
IMPORTANCE["cumulative_share_pct"] = IMPORTANCE.gain_share_pct.cumsum()


def feature_family(name):
    if name.startswith("HubTrail") or name.startswith("HubTrend"):
        return "trailing-window history"
    if name.startswith("Hub") and name in HISTORY_FEATURES:
        return "hub history"
    if name.startswith("Net"):
        return "network shape"
    if name in ("PromoActive", "PromoRunLength", "DaysSincePromo", "DaysUntilPromo",
                "PromoYesterday", "PromoTomorrow"):
        return "promotion calendar"
    if name in ("IsOpen", "OpenRunLength", "ClosedRunLength", "DaysSinceClosed",
                "DaysUntilClosed", "ClosedYesterday", "ClosedTomorrow"):
        return "operating status"
    if name.startswith(("Competitor", "Loyalty", "IsLoyalty", "HubFormat", "AssortmentTier", "LogCompetitor")):
        return "hub metadata"
    return "calendar"


IMPORTANCE["family"] = IMPORTANCE.feature.map(feature_family)
IMPORTANCE.to_csv(OUT.metrics / "feature_importance.csv", index=False)
FAMILY = IMPORTANCE.groupby("family").gain_share_pct.sum().sort_values(ascending=False)

seed_matrix = np.vstack([res["preds"] for res in STAGE_B.values()])
corr = np.corrcoef(seed_matrix)
AGREEMENT = {
    "n_models": int(seed_matrix.shape[0]),
    "mean_pairwise_correlation": float(corr[np.triu_indices_from(corr, 1)].mean()),
    "min_pairwise_correlation": float(corr[np.triu_indices_from(corr, 1)].min()),
    "mean_prediction_sd_log": float(seed_matrix.std(axis=0).mean()),
    "p95_prediction_sd_log": float(np.percentile(seed_matrix.std(axis=0), 95)),
    "gain_share_by_family": {k: float(v) for k, v in FAMILY.items()},
}
(OUT.metrics / "model_agreement.json").write_text(json.dumps(AGREEMENT, indent=2), encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top = IMPORTANCE.head(28).iloc[::-1]
axes[0].barh(top.feature, top.gain_share_pct, xerr=100 * top.gain_std / IMPORTANCE.gain_mean.sum(),
             color="#2b6cb0")
axes[0].set_title(f"Top 28 features by mean gain share (n={CFG.N_SEEDS} boosters)")
axes[0].set_xlabel("% of total gain")
axes[0].tick_params(axis="y", labelsize=8)

axes[1].bar(FAMILY.index, FAMILY.values, color="#805ad5")
axes[1].set_title("Gain share by feature family")
axes[1].set_ylabel("% of total gain")
axes[1].tick_params(axis="x", rotation=30)
for i, v in enumerate(FAMILY.values):
    axes[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)

save_fig(fig, "feature_importance")

LOG(f"ensemble agreement: mean pairwise r = {AGREEMENT['mean_pairwise_correlation']:.5f}, "
    f"mean per-row sd = {AGREEMENT['mean_prediction_sd_log']:.4f} log-units")
LOG("top-10 features: " + ", ".join(IMPORTANCE.feature.head(10)))
display(IMPORTANCE.head(20))

## 15. Error analysis

* **Purpose:** identify *which* rows the model gets wrong and whether the failures are
  systematic (fixable) or idiosyncratic (irreducible).
* **What it does:**
  1. Ranks the 25 worst rows and the 15 worst hubs by RMSLE, and characterises them against the
     network (history length, hub size, renovation status, promo share).
  2. Splits errors into over- and under-forecasts and compares their segment composition.
  3. Plots actual vs predicted daily series for the three worst hubs and three median hubs, so
     the failure mode is visible rather than inferred.
  4. Tests the specific hypotheses this dataset invites: are renovation-gap hubs worse? are
     hubs that reopened recently worse? is the first week after a long closure worse?
* **Inputs:** `EVAL`, `PER_HUB`. **Outputs:** `metrics/error_analysis.json`,
  `metrics/worst_rows.csv`, `metrics/worst_hubs.csv`, `figures/error_analysis*.png`.

In [ ]:
worst_rows = (EVAL.sort_values("abs_log_error", ascending=False)
              .head(25)[["HubID", "Date", "Weekday", "IsOpen", "PromoActive", "SchoolClosureFlag",
                         "actual", "pred", "log_error", "HorizonDays", "HubLogMean"]])
worst_rows.to_csv(OUT.metrics / "worst_rows.csv", index=False)

worst_hubs = PER_HUB.head(15).merge(
    hub_meta_clean[["HubID", "HubFormat", "AssortmentTier", "CompetitorDistance", "LoyaltyProgram"]],
    on="HubID", how="left")
worst_hubs["renovation_gap"] = worst_hubs.HubID.isin(RENOVATION_HUBS)
worst_hubs["history_days"] = worst_hubs.HubID.map(train_raw.groupby("HubID").size())
worst_hubs.to_csv(OUT.metrics / "worst_hubs.csv", index=False)

over = EVAL[EVAL.log_error > 0]
under = EVAL[EVAL.log_error < 0]
reno_mask = EVAL.Renovated == 1
first_week_after_closure = EVAL[(EVAL.IsOpen == 1) & (EVAL.DaysSinceClosed.between(1, 7))]
normal_open = EVAL[(EVAL.IsOpen == 1) & (EVAL.DaysSinceClosed > 7)]

ERROR_ANALYSIS = {
    "rows_over_forecast_pct": float(100 * len(over) / len(EVAL)),
    "rows_under_forecast_pct": float(100 * len(under) / len(EVAL)),
    "mean_log_error_over": float(over.log_error.mean()),
    "mean_log_error_under": float(under.log_error.mean()),
    "worst_1pct_rows_share_of_squared_error": float(
        100 * np.sort(EVAL.sq_log_error.to_numpy())[::-1][:max(1, len(EVAL) // 100)].sum()
        / EVAL.sq_log_error.sum()),
    "rmsle_renovation_hubs": float(np.sqrt(EVAL.loc[reno_mask, "sq_log_error"].mean())) if reno_mask.any() else None,
    "rmsle_other_hubs": float(np.sqrt(EVAL.loc[~reno_mask, "sq_log_error"].mean())),
    "rmsle_first_week_after_closure": float(np.sqrt(first_week_after_closure.sq_log_error.mean())),
    "rmsle_settled_open_days": float(np.sqrt(normal_open.sq_log_error.mean())),
    "rmsle_promo_days": float(np.sqrt(EVAL.loc[EVAL.PromoActive == 1, "sq_log_error"].mean())),
    "rmsle_non_promo_days": float(np.sqrt(EVAL.loc[EVAL.PromoActive == 0, "sq_log_error"].mean())),
    "closed_rows_all_predicted_zero": bool((EVAL.loc[EVAL.IsOpen == 0, "pred"] == 0).all()),
    "closed_rows_all_actually_zero": bool((EVAL.loc[EVAL.IsOpen == 0, "actual"] == 0).all()),
    "worst_hub_ids": [int(h) for h in PER_HUB.HubID.head(15)],
    "worst_hub_rmsle": [float(v) for v in PER_HUB.rmsle.head(15)],
}
(OUT.metrics / "error_analysis.json").write_text(json.dumps(ERROR_ANALYSIS, indent=2), encoding="utf-8")

for _k, _v in ERROR_ANALYSIS.items():
    if isinstance(_v, float):
        LOG(f"  {_k:<45} {_v:.5f}")
    elif isinstance(_v, bool):
        LOG(f"  {_k:<45} {_v}")
LOG(f"the worst 1% of rows account for "
    f"{ERROR_ANALYSIS['worst_1pct_rows_share_of_squared_error']:.1f}% of total squared log error")
display(worst_hubs)
display(worst_rows)

### 15.1 Worst and typical hubs, visualised

* **Purpose:** make the failure modes concrete by plotting the daily actual and predicted series
  of the three hubs with the highest validation RMSLE alongside three median-performing hubs.
* **What it does:** for each selected hub, draws the full validation window with markers for
  promo days and closed days, plus the hub's trailing 28-day mean level from the statistics
  window as a reference line.
* **Inputs:** `EVAL`, `PER_HUB`. **Outputs:** `figures/error_analysis_hub_series.png`.
* **Reading the plot:** a flat prediction against a shifted actual level means the hub's level
  moved after the statistics window closed — a genuine limit of a 42-day-ahead forecast.
  Wrong weekday shape or missed promo spikes would instead point to fixable feature gaps.

In [ ]:
median_pos = len(PER_HUB) // 2
selected = list(PER_HUB.HubID.head(3)) + list(PER_HUB.HubID.iloc[median_pos:median_pos + 3])
labels = ["worst"] * 3 + ["median"] * 3

fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharex=True)
for ax, hub, label in zip(axes.ravel(), selected, labels):
    sub = EVAL[EVAL.HubID == hub].sort_values("Date")
    ax.plot(sub.Date, sub.actual, color="#1a202c", lw=1.6, marker="o", ms=3, label="actual")
    ax.plot(sub.Date, sub.pred, color="#c05621", lw=1.6, ls="--", marker="s", ms=3, label="predicted")
    promo = sub[sub.PromoActive == 1]
    ax.scatter(promo.Date, promo.actual, color="#2f855a", s=28, zorder=5, label="promo day")
    closed = sub[sub.IsOpen == 0]
    ax.scatter(closed.Date, np.zeros(len(closed)), color="#a0aec0", marker="x", s=28, label="closed")
    ax.axhline(np.expm1(sub.HubTrailMean28.iloc[0]), color="#2b6cb0", lw=1, ls=":",
               label="trailing 28d level")
    ax.set_title(f"Hub {int(hub)} ({label}) — RMSLE "
                 f"{float(PER_HUB.loc[PER_HUB.HubID == hub, 'rmsle'].iloc[0]):.3f}", fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0, 0].legend(fontsize=7, loc="upper left")
fig.suptitle("Validation-window forecasts: three worst hubs (top) vs three median hubs (bottom)")
save_fig(fig, "error_analysis_hub_series")

## 16. Stage C — final training and test inference

* **Purpose:** refit the chosen configuration on **all** available history and predict the
  competition's test window.
* **What it does:** trains `CFG.N_SEEDS` boosters on the `final` block — whose statistics window
  ends on 2015-06-19, the last day of `orders_train.csv`, and whose rolling-origin training rows
  run up to that same day — then averages their log-space predictions for the 46,830 test rows.
  Jobs are dealt across both T4s exactly as in Stage B.
* **Round budget:** `BEST_ROUNDS x CFG.FINAL_ROUND_INFLATION`. The final model sees the
  additional 42 days of history that the validation fit did not, so a modest increase in
  capacity is appropriate; there is no held-out window left to early-stop against, and
  inflating by a fixed, disclosed factor is safer than guessing a larger number.
* **Inputs:** `cache/final.parquet`. **Outputs:** `STAGE_C`, `TEST_LOG_PRED`,
  `predictions/test_predictions_per_model.csv`, `metrics/stage_c_final.json`, boosters in
  `models/`.
* **Reproducibility:** seeds are `CFG.SEED + i`; every booster, its parameters and its
  importances are saved to disk and packaged in the final ZIP.

In [ ]:
FINAL_ROUNDS = int(max(200, round(BEST_ROUNDS * CFG.FINAL_ROUND_INFLATION)))

stage_c_jobs = [
    make_job(tag=f"C_final_seed{CFG.SEED + i}", block="final", seed=CFG.SEED + i,
             evaluate=False, num_boost_round=FINAL_ROUNDS,
             use_weights=BEST_WEIGHTING, early_stopping=None)
    for i in range(CFG.N_SEEDS)
]

STAGE_C = run_jobs(stage_c_jobs, desc="stage C (final fit)")

FINAL_TARGET = BLOCK_DATA["final"].query("role == 'target'").reset_index(drop=True)
assert len(FINAL_TARGET) == len(test_raw)

model_preds = {tag: res["preds"] for tag, res in sorted(STAGE_C.items())}
for tag, p in model_preds.items():
    assert len(p) == len(FINAL_TARGET), f"{tag}: prediction length mismatch"
TEST_LOG_PRED = np.mean(list(model_preds.values()), axis=0)
TEST_RAW_PRED = postprocess(TEST_LOG_PRED, FINAL_TARGET.IsOpen.to_numpy(), CALIB_MULTIPLIER)

per_model = pd.DataFrame({"Id": FINAL_TARGET.Id.to_numpy()})
for tag, p in model_preds.items():
    per_model[tag] = np.expm1(p)
per_model["ensemble_log_mean"] = TEST_LOG_PRED
per_model["final_prediction"] = TEST_RAW_PRED
per_model.to_csv(OUT.preds / "test_predictions_per_model.csv", index=False)

spread = np.vstack(list(model_preds.values())).std(axis=0)
STAGE_C_SUMMARY = {
    "rounds": FINAL_ROUNDS,
    "rounds_source": f"{BEST_ROUNDS} (back-test early stopping) x {CFG.FINAL_ROUND_INFLATION}",
    "n_models": len(model_preds),
    "seeds": [int(res["meta"]["seed"]) for res in STAGE_C.values()],
    "devices": sorted({res["meta"]["device"] for res in STAGE_C.values()}),
    "train_rows": int(BLOCK_META["final"]["train_rows"]),
    "seconds_total": float(sum(res["meta"]["seconds"] for res in STAGE_C.values())),
    "calibration_multiplier": CALIB_MULTIPLIER,
    "ensemble_sd_log_mean": float(spread.mean()),
    "ensemble_sd_log_p95": float(np.percentile(spread, 95)),
    "prediction_summary": {k: float(v) for k, v in pd.Series(TEST_RAW_PRED).describe().items()},
    "zero_predictions": int((TEST_RAW_PRED == 0).sum()),
    "closed_test_rows": int((FINAL_TARGET.IsOpen == 0).sum()),
}
(OUT.metrics / "stage_c_final.json").write_text(json.dumps(STAGE_C_SUMMARY, indent=2), encoding="utf-8")

LOG(f"final ensemble: {len(model_preds)} boosters x {FINAL_ROUNDS} rounds on "
    f"{STAGE_C_SUMMARY['train_rows']:,} rows")
LOG(f"test predictions: mean {TEST_RAW_PRED.mean():.1f}, median {np.median(TEST_RAW_PRED):.1f}, "
    f"max {TEST_RAW_PRED.max():.0f}, zeros {STAGE_C_SUMMARY['zero_predictions']:,}")
LOG(f"ensemble disagreement (log sd): mean {spread.mean():.4f}, p95 {np.percentile(spread, 95):.4f}")
display(per_model.head())

### 16.1 Sanity-checking the test predictions against history

* **Purpose:** catch a silent catastrophe (wrong join, scale error, all-zero column) *before* the
  submission is written, by comparing the forecast distribution with the recent past.
* **What it does:** compares the predicted test distribution against the last 42 days of actuals
  on identical terms, per-hub mean predicted vs per-hub recent mean actual, the predicted daily
  network curve appended to history, and the predicted weekday profile against the historical
  one. It then asserts hard bounds: every prediction finite and non-negative, closed rows exactly
  zero, per-hub mean forecast within 0.4–2.5x of that hub's recent mean for at least 99% of hubs.
* **Inputs:** `TEST_RAW_PRED`. **Outputs:** `figures/test_prediction_sanity.png`,
  `metrics/test_prediction_checks.json`.
* **Why these bounds:** a 42-day-ahead forecast should not reshape the network. Anything outside
  them indicates a bug, not a legitimate forecast.

In [ ]:
TEST_DF = FINAL_TARGET[["Id", "HubID", "Date", "Weekday", "IsOpen", "PromoActive"]].copy()
TEST_DF["pred"] = TEST_RAW_PRED

recent = train_raw[train_raw.Date > TRAIN_MAX_DATE - pd.Timedelta(days=CFG.VALID_DAYS)]
recent_open = recent[recent.IsOpen == 1]
pred_open = TEST_DF[TEST_DF.IsOpen == 1]

hub_recent = recent_open.groupby("HubID").OrderVolume.mean()
hub_pred = pred_open.groupby("HubID").pred.mean()
ratio = (hub_pred / hub_recent).dropna()

CHECKS = {
    "all_finite": bool(np.isfinite(TEST_RAW_PRED).all()),
    "all_non_negative": bool((TEST_RAW_PRED >= 0).all()),
    "closed_rows_are_zero": bool((TEST_DF.loc[TEST_DF.IsOpen == 0, "pred"] == 0).all()),
    "open_rows_all_positive": bool((pred_open.pred > 0).all()),
    "pred_mean_open": float(pred_open.pred.mean()),
    "recent_actual_mean_open": float(recent_open.OrderVolume.mean()),
    "mean_ratio_pred_over_recent": float(pred_open.pred.mean() / recent_open.OrderVolume.mean()),
    "hub_ratio_p01": float(ratio.quantile(0.01)),
    "hub_ratio_p50": float(ratio.quantile(0.50)),
    "hub_ratio_p99": float(ratio.quantile(0.99)),
    "hubs_within_0.4_to_2.5x": float((ratio.between(0.4, 2.5)).mean()),
    "max_prediction": float(TEST_RAW_PRED.max()),
    "historical_max": float(train_raw.OrderVolume.max()),
}
(OUT.metrics / "test_prediction_checks.json").write_text(json.dumps(CHECKS, indent=2), encoding="utf-8")

assert CHECKS["all_finite"] and CHECKS["all_non_negative"], "non-finite or negative predictions"
assert CHECKS["closed_rows_are_zero"], "a closed test row received a non-zero prediction"
assert CHECKS["open_rows_all_positive"], "an open test row received a zero prediction"
assert CHECKS["hubs_within_0.4_to_2.5x"] >= 0.99, (
    f"only {CHECKS['hubs_within_0.4_to_2.5x']:.3f} of hubs forecast within 0.4-2.5x of their "
    "recent mean — investigate before submitting")
assert CHECKS["max_prediction"] <= 3 * CHECKS["historical_max"], "implausibly large prediction"

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].hist(np.log1p(recent_open.OrderVolume), bins=80, alpha=0.6, label="last 42 days actual",
                color="#2b6cb0", density=True)
axes[0, 0].hist(np.log1p(pred_open.pred), bins=80, alpha=0.6, label="test forecast",
                color="#c05621", density=True)
axes[0, 0].set_title("Distribution check (open rows, log1p scale)")
axes[0, 0].legend(fontsize=8)

axes[0, 1].scatter(hub_recent.reindex(hub_pred.index), hub_pred, s=6, alpha=0.5, color="#2b6cb0")
_lim = [hub_recent.min(), hub_recent.max()]
axes[0, 1].plot(_lim, _lim, color="#c05621", lw=1.5)
axes[0, 1].set_xscale("log"); axes[0, 1].set_yscale("log")
axes[0, 1].set_xlabel("hub mean, last 42 days (actual)")
axes[0, 1].set_ylabel("hub mean, test window (forecast)")
axes[0, 1].set_title(f"Per-hub level check (median ratio {ratio.median():.3f})")

hist_daily = train_raw.groupby("Date").OrderVolume.sum() / 1e6
pred_daily = TEST_DF.groupby("Date").pred.sum() / 1e6
axes[1, 0].plot(hist_daily.index[-180:], hist_daily.values[-180:], color="#1a202c", lw=1,
                label="history")
axes[1, 0].plot(pred_daily.index, pred_daily.values, color="#c05621", lw=1.5, label="forecast")
axes[1, 0].axvline(TEST_MIN_DATE, ls="--", color="#718096", lw=1)
axes[1, 0].set_title("Network daily volume: last 180 days of history + 42-day forecast")
axes[1, 0].set_ylabel("millions of orders")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].legend(fontsize=8)

hist_dow = recent_open.groupby("Weekday").OrderVolume.mean()
pred_dow = pred_open.groupby("Weekday").pred.mean()
w = 0.4
axes[1, 1].bar(hist_dow.index - w / 2, hist_dow.values, width=w, label="recent actual", color="#2b6cb0")
axes[1, 1].bar(pred_dow.index + w / 2, pred_dow.reindex(hist_dow.index).values, width=w,
               label="forecast", color="#c05621")
axes[1, 1].set_title("Weekday profile check (open rows)")
axes[1, 1].set_xlabel("Weekday (1=Mon .. 7=Sun)")
axes[1, 1].legend(fontsize=8)

save_fig(fig, "test_prediction_sanity")
for _k, _v in CHECKS.items():
    LOG(f"  check {_k:<32} {_v}")

## 17. Submission generation

* **Purpose:** write `submission.csv` in exactly the format `sample_submission.csv` specifies:
  two columns, `Id` and `OrderVolume`, one row per test `Id`, in the sample file's row order.
* **What it does:** joins the predictions onto `sample_submission.csv` by `Id` (a left join on
  the official file, so the order and the row set come from the competition, not from our
  pipeline), casts predictions to integers, and asserts the result matches the template's
  shape, columns, dtype and `Id` ordering before writing.
* **Inputs:** `TEST_DF`, `sample_sub`. **Outputs:** `submission/submission.csv` plus a copy at
  the working-directory root (where Kaggle's "Submit" button looks), and
  `predictions/test_predictions_detailed.csv` with the full per-row context for analysis.
* **Rounding:** `OrderVolume` is a count, so predictions are rounded to the nearest integer. The
  RMSLE impact is negligible (well under 1e-5) and the output matches the target's domain.

In [ ]:
submission = sample_sub[[CFG.ID_COL]].merge(
    TEST_DF[["Id", "pred"]].rename(columns={"pred": CFG.TARGET}), on=CFG.ID_COL, how="left")
submission[CFG.TARGET] = submission[CFG.TARGET].astype("int64")

assert len(submission) == len(sample_sub), "row count differs from the sample submission"
assert list(submission.columns) == [CFG.ID_COL, CFG.TARGET], list(submission.columns)
assert submission[CFG.ID_COL].tolist() == sample_sub[CFG.ID_COL].tolist(), "Id order changed"
assert submission[CFG.TARGET].notna().all(), "a test Id received no prediction"
assert (submission[CFG.TARGET] >= 0).all(), "negative prediction in the submission"

SUB_PATH = OUT.submission / "submission.csv"
submission.to_csv(SUB_PATH, index=False)
ROOT_SUB = WORK_DIR / "submission.csv"
submission.to_csv(ROOT_SUB, index=False)

detailed = TEST_DF.merge(
    hub_meta_clean[["HubID", "HubFormat", "AssortmentTier", "LoyaltyProgram"]], on="HubID", how="left")
detailed["ensemble_sd_log"] = spread
detailed["log_prediction"] = TEST_LOG_PRED
detailed.to_csv(OUT.preds / "test_predictions_detailed.csv", index=False)

LOG(f"submission written -> {SUB_PATH} ({SUB_PATH.stat().st_size / 1e6:.2f} MB)")
LOG(f"kaggle-root copy    -> {ROOT_SUB}")
LOG(f"rows {len(submission):,} | mean {submission[CFG.TARGET].mean():.1f} | "
    f"zeros {(submission[CFG.TARGET] == 0).sum():,}")
display(submission.head())
display(submission.describe().T)

## 18. Output verification

* **Purpose:** re-read the submission from disk and validate it as an independent auditor would,
  rather than trusting the in-memory frame that produced it.
* **What it does:** reloads `submission.csv` with plain `pd.read_csv`, then checks: exact column
  names and order, integer dtype, row count, `Id` set equality with `orders_test.csv`, no nulls,
  no negatives, no duplicate `Id`s, zero predictions occurring **exactly** on the closed test
  rows, and a final RMSLE self-check that confirms the metric implementation reproduces a known
  value on synthetic data.
* **Inputs:** the files on disk. **Outputs:** `metrics/submission_verification.json`. Any
  failure raises.

In [ ]:
reloaded = pd.read_csv(SUB_PATH)
closed_ids = set(TEST_DF.loc[TEST_DF.IsOpen == 0, "Id"])
zero_ids = set(reloaded.loc[reloaded[CFG.TARGET] == 0, CFG.ID_COL])

# independent check that rmsle() is implemented correctly
_a = np.array([0.0, 1.0, 10.0, 100.0])
_p = np.array([0.0, 2.0, 9.0, 110.0])
_expected = float(np.sqrt(np.mean((np.log1p(_p) - np.log1p(_a)) ** 2)))

VERIFICATION = {
    "file": str(SUB_PATH),
    "size_bytes": int(SUB_PATH.stat().st_size),
    "columns_exact": list(reloaded.columns) == [CFG.ID_COL, CFG.TARGET],
    "dtype_integer": str(reloaded[CFG.TARGET].dtype).startswith("int"),
    "row_count_matches_test": len(reloaded) == len(test_raw),
    "id_set_matches_test": set(reloaded[CFG.ID_COL]) == set(test_raw[CFG.ID_COL]),
    "id_order_matches_sample": reloaded[CFG.ID_COL].tolist() == sample_sub[CFG.ID_COL].tolist(),
    "no_nulls": bool(reloaded.notna().all().all()),
    "no_negatives": bool((reloaded[CFG.TARGET] >= 0).all()),
    "no_duplicate_ids": bool(not reloaded[CFG.ID_COL].duplicated().any()),
    "zeros_exactly_match_closed_rows": zero_ids == closed_ids,
    "n_zero_predictions": int(len(zero_ids)),
    "n_closed_test_rows": int(len(closed_ids)),
    "rmsle_implementation_selfcheck_ok": abs(rmsle(_a, _p) - _expected) < 1e-12,
    "validation_rmsle_expected_on_leaderboard": VALID_RMSLE_CAL,
    "backtest_rmsle": float(stage_a_df.iloc[0].backtest_rmsle_all_rows),
}
(OUT.metrics / "submission_verification.json").write_text(json.dumps(VERIFICATION, indent=2), encoding="utf-8")

failed = [k for k, v in VERIFICATION.items() if isinstance(v, bool) and not v]
for _k, _v in VERIFICATION.items():
    LOG(f"  verify {_k:<45} {_v}")
if failed:
    raise AssertionError("submission verification failed: " + ", ".join(failed))
LOG("submission verification: ALL CHECKS PASSED")

## 19. Reproducibility record and written summary

* **Purpose:** satisfy `Rules.md` #5–#7 — a run that reproduces end-to-end, disclosed seeds, and
  a 1–2 page written summary of approach, validation and findings.
* **What it does:** writes `reports/run_config.json` (every configuration value, the resolved
  feature list, block definitions, chosen hyper-parameters and library versions) and
  `reports/summary.md`, a self-contained write-up generated from the numbers this run actually
  produced — no hard-coded results.
* **Inputs:** every metrics object computed above. **Outputs:** `reports/run_config.json`,
  `reports/summary.md` (also rendered inline below).

In [ ]:
RUN_CONFIG = {
    "run_tag": RUN_TAG,
    "seed": CFG.SEED,
    "seeds_used": [CFG.SEED + i for i in range(CFG.N_SEEDS)],
    "config": {k: (list(v) if isinstance(v, tuple) else v)
               for k, v in vars(CFG).items() if not k.startswith("_")},
    "chosen": {"use_recency_weights": BEST_WEIGHTING,
               "validation_rounds": BEST_ROUNDS,
               "final_rounds": FINAL_ROUNDS,
               "calibration_multiplier": CALIB_MULTIPLIER},
    "blocks": {k: {kk: (str(vv.date()) if isinstance(vv, pd.Timestamp) else vv)
                   for kk, vv in v.items()} for k, v in BLOCKS.items()},
    "features": FEATURES,
    "environment": ENVIRONMENT,
    "results": {"validation_rmsle_uncalibrated": VALID_RMSLE,
                "validation_rmsle_calibrated": VALID_RMSLE_CAL,
                "backtest_rmsle": float(stage_a_df.iloc[0].backtest_rmsle_all_rows),
                "best_baseline_rmsle": STAGE_B_SUMMARY["best_baseline_rmsle"]},
}
(OUT.reports / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2, default=str), encoding="utf-8")

top_feats = ", ".join(f"`{f}`" for f in IMPORTANCE.feature.head(8))
fam_lines = "\n".join(f"| {k} | {v:.1f}% |" for k, v in AGREEMENT["gain_share_by_family"].items())
base_lines = "\n".join(
    f"| {k.split(':')[1]} | {v:.5f} |" for k, v in BASELINES.items() if k.startswith("valid:"))
hz = SEGMENT_METRICS.query("segment == 'HorizonWeek'").sort_values("value")
hz_lines = "\n".join(f"| week {int(r.value)} | {r.rmsle:.5f} |" for r in hz.itertuples())

SUMMARY_MD = f"""# Solari hub demand forecasting — approach, validation and findings

**Run:** `{RUN_TAG}` · **Seed:** `{CFG.SEED}` (models use `{CFG.SEED}`..`{CFG.SEED + CFG.N_SEEDS - 1}`)
· **Hardware:** {", ".join(ENVIRONMENT["devices"])} · **XGBoost** {ENVIRONMENT["xgboost"]}

## 1. Problem and metric

Forecast daily `OrderVolume` for {train_raw.HubID.nunique():,} hubs over
{TEST_HORIZON_DAYS} future days ({TEST_MIN_DATE.date()} to {TEST_MAX_DATE.date()}), scored with
RMSLE. Hub scale spans {TARGET_SUMMARY["hub_mean_ratio_max_over_min"]:.0f}x from smallest to
largest, so the model is trained directly on `log1p(OrderVolume)` with an L2 objective — the
loss the metric implies — and predictions are mapped back with `expm1`.

## 2. Approach

* **Model:** {CFG.N_SEEDS} XGBoost boosters (`hist`, depth {CFG.XGB_PARAMS["max_depth"]},
  eta {CFG.XGB_PARAMS["eta"]}), {FINAL_ROUNDS} rounds, averaged in log space. No deep learning.
* **Rows used:** open days with positive volume. Closed days (`IsOpen == 0`) are predicted as
  exact zeros — in the full history that implication holds without a single exception.
* **Features ({len(FEATURES)} total):** hub history statistics (overall, by weekday, by promo
  state, by month, with shrinkage), trailing 7/28/91/182/365-day levels and trends, the
  operating calendar supplied for the test window (promo runs, days since/until promo, closure
  runs), calendar seasonality, and hub metadata (format, assortment, competitor distance and
  age, loyalty programme timing). `AppSessions` exists only in train, so it enters solely as
  hub-level aggregates computed inside each statistics window.
* **Leakage control:** history statistics are always computed from a window ending strictly
  before the window being predicted, and training rows come from *rolling origins* —
  {BLOCK_META["final"]["train_origins"]} consecutive {CFG.VALID_DAYS}-day windows, each described
  by its own earlier statistics window.
  No training row contributes to any statistic that describes it. Three automated audits assert
  this (Section 8.2).
* **Calibration:** one global multiplier ({CALIB_MULTIPLIER}), selected on the validation block
  and confirmed on an independent back-test block.

## 3. Validation

Time-blocked, never random. Two held-out *future* blocks of exactly {CFG.VALID_DAYS} days —
the same horizon as the leaderboard window:

| Block | Target window | Purpose |
| --- | --- | --- |
| back-test | {BLOCKS["backtest"]["target_start"].date()} → {BLOCKS["backtest"]["target_end"].date()} | hyper-parameter and round selection |
| validation | {BLOCKS["valid"]["target_start"].date()} → {BLOCKS["valid"]["target_end"].date()} | unbiased scoring (fixed round budget) |

**Results**

| Model | RMSLE |
| --- | --- |
| single booster (mean of {CFG.N_SEEDS}) | {STAGE_B_SUMMARY["mean_single_model_rmsle"]:.5f} |
| {CFG.N_SEEDS}-booster ensemble | {VALID_RMSLE:.5f} |
| ensemble + calibration | **{VALID_RMSLE_CAL:.5f}** |
| back-test block (independent) | {float(stage_a_df.iloc[0].backtest_rmsle_all_rows):.5f} |

**Naive baselines on the same validation block**

| Baseline | RMSLE |
| --- | --- |
{base_lines}

**Accuracy decay over the forecast horizon**

| Horizon | RMSLE |
| --- | --- |
{hz_lines}

## 4. Findings

* Hub identity and recent level dominate: the top features are {top_feats}.
* Gain share by feature family:

| Family | Share |
| --- | --- |
{fam_lines}

* Promotions lift demand materially and the model captures them: RMSLE on promo days is
  {ERROR_ANALYSIS["rmsle_promo_days"]:.4f} vs {ERROR_ANALYSIS["rmsle_non_promo_days"]:.4f} on
  non-promo days.
* Error is concentrated: the worst 1% of rows carry
  {ERROR_ANALYSIS["worst_1pct_rows_share_of_squared_error"]:.1f}% of the total squared log error.
* Hubs with a renovation gap in their history score
  {ERROR_ANALYSIS["rmsle_renovation_hubs"]:.4f} vs {ERROR_ANALYSIS["rmsle_other_hubs"]:.4f} for
  the rest, and the first week after a closure is harder
  ({ERROR_ANALYSIS["rmsle_first_week_after_closure"]:.4f}) than settled trading days
  ({ERROR_ANALYSIS["rmsle_settled_open_days"]:.4f}).
* Residual log-space bias after calibration is
  {OVERALL["log_bias"]:+.5f}, i.e. the forecast is effectively unbiased at the network level.

## 5. Limitations

* `RegionalHoliday` takes values 1–3 in training but is uniformly 0 across the test window, so
  the holiday features cannot contribute there; the validation block does contain holidays, so
  the reported score is, if anything, conservative for this specific test period.
* A 42-day-ahead forecast cannot react to level shifts that begin after the history ends; the
  hub-series plots in Section 15.1 show this is the dominant residual failure mode.
* The calibration multiplier is a single parameter fitted on the validation block, which makes
  the reported validation RMSLE marginally optimistic. Its optimum agrees with the independent
  back-test block, which is the check that keeps it honest.
"""

(OUT.reports / "summary.md").write_text(SUMMARY_MD, encoding="utf-8")
LOG(f"summary written -> {OUT.reports / 'summary.md'} ({len(SUMMARY_MD)} chars)")

from IPython.display import Markdown, display as _display
_display(Markdown(SUMMARY_MD))

## 20. Final artifact packaging

This is the last cell. It verifies that everything the run was supposed to produce exists, then
packages it into a **single ZIP archive** that can be downloaded from Kaggle's *Output* pane.

* **Purpose:** one click, everything useful, nothing wasteful.
* **What it does:**
  1. **Verifies expected outputs.** A manifest of required artefacts (submission, per-model and
     detailed predictions, every metrics JSON/CSV, every figure, the summary report, the run
     config, the worker logs, the trained boosters) is checked against the filesystem. Missing
     *required* items raise; missing optional items are reported.
  2. **Excludes junk.** The intermediate `cache/` parquet files and worker job files are large
     and fully reproducible from the notebook, so they are skipped, as are `__pycache__`,
     `.ipynb_checkpoints` and any zero-byte file. Stage A/B boosters are also skipped — they
     exist only to choose hyper-parameters and to score the validation block, and their
     metadata, importances and predictions are archived in full; only the Stage C models that
     produced the submission are shipped as binaries.
  3. **Writes the archive** to `solari_forecast_<run tag>.zip` at the working-directory root,
     with deflate compression, and re-opens it to test its integrity (`ZipFile.testzip()`).
  4. **Prints the location, size and a grouped manifest** of its contents, plus the headline
     metrics, so the run ends with a self-contained report.
* **Inputs:** everything under `OUT.root`. **Outputs:** the ZIP, `reports/manifest.csv`, and a
  printed summary.
* **Note:** `submission.csv` is also left uncompressed at the working-directory root, because
  Kaggle's competition submission flow expects it there — the ZIP is for the artefacts.

In [ ]:
import zipfile

REQUIRED = [
    OUT.submission / "submission.csv",
    OUT.preds / "test_predictions_per_model.csv",
    OUT.preds / "test_predictions_detailed.csv",
    OUT.metrics / "environment.json",
    OUT.metrics / "data_validation.json",
    OUT.metrics / "validation_blocks.json",
    OUT.metrics / "block_datasets.json",
    OUT.metrics / "leakage_audit.json",
    OUT.metrics / "baselines.json",
    OUT.metrics / "stage_a_ablation.json",
    OUT.metrics / "stage_b_validation.json",
    OUT.metrics / "calibration.json",
    OUT.metrics / "validation_overall.json",
    OUT.metrics / "validation_segments.csv",
    OUT.metrics / "validation_per_hub.csv",
    OUT.metrics / "feature_importance.csv",
    OUT.metrics / "error_analysis.json",
    OUT.metrics / "stage_c_final.json",
    OUT.metrics / "test_prediction_checks.json",
    OUT.metrics / "submission_verification.json",
    OUT.reports / "summary.md",
    OUT.reports / "run_config.json",
    OUT.logs / "run.log",
    OUT.figures / "diagnostics_validation.png",
    OUT.figures / "feature_importance.png",
    OUT.figures / "test_prediction_sanity.png",
]
REQUIRED += [OUT.models / f"C_final_seed{CFG.SEED + i}.ubj" for i in range(CFG.N_SEEDS)]

missing = [str(p) for p in REQUIRED if not Path(p).exists()]
if missing:
    raise FileNotFoundError("expected artefacts are missing:\n  - " + "\n  - ".join(missing))
LOG(f"verified {len(REQUIRED)} required artefacts are present")

EXCLUDE_DIRS = {"cache", "__pycache__", ".ipynb_checkpoints"}
EXCLUDE_SUFFIX = {".parquet", ".pyc"}

entries = []
for path in sorted(OUT.root.rglob("*")):
    if not path.is_file():
        continue
    rel = path.relative_to(OUT.root)
    if set(rel.parts[:-1]) & EXCLUDE_DIRS or path.suffix in EXCLUDE_SUFFIX:
        continue
    if rel.name.startswith("jobs_") and rel.suffix == ".json":
        continue
    # tuning/validation boosters are large and reproducible; only the final models are shipped
    # (their metadata, importances and predictions are kept for every stage)
    if rel.suffix == ".ubj" and not rel.name.startswith("C_final"):
        continue
    if path.stat().st_size == 0:
        LOG(f"skipping zero-byte file {rel}", "WARN")
        continue
    entries.append((path, rel))

ZIP_PATH = WORK_DIR / f"solari_forecast_{RUN_TAG}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path, rel in tqdm(entries, desc="packaging artefacts", unit="file"):
        zf.write(path, arcname=str(Path("solari_forecast") / rel))

with zipfile.ZipFile(ZIP_PATH) as zf:
    broken = zf.testzip()
    infos = zf.infolist()
if broken is not None:
    raise RuntimeError(f"archive integrity check failed on {broken}")

manifest = pd.DataFrame([
    {"path": i.filename,
     "group": (Path(i.filename).parts[1] if len(Path(i.filename).parts) > 2 else "root"),
     "size_kb": round(i.file_size / 1024, 1),
     "compressed_kb": round(i.compress_size / 1024, 1)}
    for i in infos]).sort_values(["group", "path"])
manifest.to_csv(OUT.reports / "manifest.csv", index=False)

group_summary = (manifest.groupby("group")
                 .agg(files=("path", "size"), size_kb=("size_kb", "sum"))
                 .sort_values("size_kb", ascending=False))

print("\n" + "=" * 78)
print(f"ARCHIVE   : {ZIP_PATH}")
print(f"SIZE      : {ZIP_PATH.stat().st_size / 1e6:.2f} MB compressed "
      f"({manifest.size_kb.sum() / 1024:.2f} MB uncompressed, {len(manifest)} files)")
print(f"DOWNLOAD  : Kaggle right-hand 'Output' pane -> /kaggle/working/{ZIP_PATH.name}")
print("=" * 78)
print("\nCONTENTS BY GROUP")
print(group_summary.to_string())
print("\nMANIFEST")
for grp, sub in manifest.groupby("group"):
    print(f"\n  [{grp}]")
    for row in sub.itertuples():
        print(f"    {row.size_kb:>9.1f} KB  {row.path}")

print("\n" + "=" * 78)
print("HEADLINE RESULTS")
print(f"  validation RMSLE (calibrated, 42-day held-out future block) : {VALID_RMSLE_CAL:.5f}")
print(f"  back-test RMSLE (independent earlier future block)          : "
      f"{float(stage_a_df.iloc[0].backtest_rmsle_all_rows):.5f}")
print(f"  best naive baseline on the validation block                 : "
      f"{STAGE_B_SUMMARY['best_baseline_rmsle']:.5f}")
print(f"  final ensemble : {CFG.N_SEEDS} XGBoost boosters x {FINAL_ROUNDS} rounds on "
      f"{', '.join(STAGE_C_SUMMARY['devices'])}")
print(f"  submission     : {SUB_PATH}  ({len(submission):,} rows)")
print(f"  master seed    : {CFG.SEED}   |   run tag: {RUN_TAG}")
print(f"  total runtime  : {(dt.datetime.now() - RUN_STARTED).total_seconds() / 60:.1f} minutes")
print("=" * 78)

LOG(f"archive written -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.2f} MB, {len(manifest)} files)")
display(group_summary)